In [ ]:
!pip install shap pvlib xgboost scikit-learn pandas matplotlib scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 141.9 MB/s eta 0:00:00


In [ ]:
!pip install pandas

In [ ]:
# ==========================================
# PM 2.5 All Parameters
# ==========================================
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import pvlib

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
start_time = time.time()
np.random.seed(1)

# Full Sensor Table from MATLAB
sensor_table = [
    ('ind01', '70b3d540f40ce423'),
    ('ind02', '70b3d540f40ce429'),
    ('ind03', '70b3d540f40ce435'),
    ('ind04', '70b3d540f40ce430'),
    ('ind06', '70b3d540f40ce43c'),
    ('ind07', '70b3d540f40ce420'),
    ('ind08', '70b3d540f40ce425'),
    ('ind09', '70b3d540f40ce438'),
    ('ind10', '70b3d540f40ce436'),
    ('ind11', '70b3d540f40ce433'),
    ('ind12', '70b3d540f40ce43a'),
    ('ind13', '70b3d540f40ce422'),
    ('ind14', '70b3d540f40ce421'),
    ('ind15', '70b3d540f40ce427'),
    ('ind16', '70b3d540f40ce426'),
    ('ind17', '70b3d540f40ce42f'),
    ('ind18', '70b3d540f40ce434'),
    ('ind19', '70b3d540f40ce424'),
    ('ind20', '70b3d540f40ce42d'),
    ('ind21', '70b3d540f40ce43b'),
]

# Runtime settings
shap_swarm_points = 1000
shap_target_bins = 10
shap_background_centroids = 200

# Geographic Coordinates for Solar Angle
lon = 96.70287
lat = 32.96735

# Storage for final CSV outputs
all_importance_list = []
all_shap_summary_list = []
all_shap_values_list = []
all_X_list = []

# Helper function to compute optimal lag via cross-correlation
def get_best_lag(x_series, y_series):
    x = (x_series - x_series.mean()) / x_series.std()
    y = (y_series - y_series.mean()) / y_series.std()
    correlation = signal.correlate(x, y, mode='full')
    lags = signal.correlation_lags(len(x), len(y), mode='full')
    best_lag = lags[np.argmax(correlation)]
    return best_lag

# Column renaming mappings
old_names = [
    'pc0_1','pc0_3','pc0_5', 'pc1_0','pc2_5','pc5_0', 'pc10_0','pm0_1', 'pm0_3', 'pm0_5',
    'pm1_0_outdoorData','pm2_5_outdoorData', 'pm5_0','pm10_0_outdoorData',
    'barrometricPressureBars','airTemperature', 'relativeHumidity','dewPoint',
    'windDirectionMagnetic','windSpeedMetersPerSecond',
    'channelA410nm', 'channelA435nm', 'channelA460nm', 'channelA485nm',
    'channelA510nm', 'channelA535nm', 'channelA560nm', 'channelA585nm',
    'channelA610nm', 'channelA645nm', 'channelA680nm', 'channelA705nm','channelA730nm',
    'channelA760nm', 'channelA810nm', 'channelA860nm', 'channelA900nm', 'channelA940nm',
    'temperature','humidity', 'co2','luminosity','pressure','pm1_0_indoorData',
    'pm2_5_indoorData','pm10_0_indoorData'
]
new_names = [
    'Outdoor PC(0.1)', 'Outdoor PC(0.3)', 'Outdoor PC(0.5)', 'Outdoor PC(1.0)', 'Outdoor PC(2.5)', 'Outdoor PC(5.0)', 'Outdoor PC(10.0)',
    'Outdoor PM(0.1)', 'Outdoor PM(0.3)', 'Outdoor PM(0.5)', 'Outdoor PM(1.0)', 'Outdoor PM(2.5)', 'Outdoor PM(5.0)', 'Outdoor PM(10.0)',
    'Outdoor Pressure', 'Outdoor Temperature', 'Outdoor Humidity', 'Outdoor Dew Point', 'Outdoor Wind Direction', 'Outdoor Wind Speed',
    '410nm', '435nm', '460nm', '485nm', '510nm', '535nm', '560nm', '585nm',
    '610nm', '645nm', '680nm', '705nm', '730nm', '760nm', '810nm', '860nm', '900nm', '940nm',
    'Indoor Temperature', 'Indoor Humidity', 'Indoor CO2', 'Indoor Luminosity', 'Indoor Pressure',
    'Indoor PM(1.0)', 'Indoor PM(2.5)', 'Indoor PM(10.0)'
]
rename_dict = dict(zip(old_names, new_names))

predictor_names = [
    "Outdoor PC(0.1)", "Outdoor PC(0.3)", "Outdoor PC(0.5)", "Outdoor PC(1.0)", "Outdoor PC(2.5)", "Outdoor PC(5.0)", "Outdoor PC(10.0)",
    "Outdoor PM(0.1)", "Outdoor PM(0.3)", "Outdoor PM(0.5)", "Outdoor PM(1.0)", "Outdoor PM(2.5)", "Outdoor PM(5.0)", "Outdoor PM(10.0)",
    "Outdoor Pressure", "Outdoor Temperature", "Outdoor Humidity", "Outdoor Dew Point", "Outdoor Wind Direction", "Outdoor Wind Speed",
    "410nm", "435nm", "460nm", "485nm", "510nm", "535nm", "560nm", "585nm", "610nm", "645nm", "680nm", "705nm",
    "730nm", "760nm", "810nm", "860nm", "900nm", "940nm", "SolarZenithAngle"
]

# ==========================================
# 2. MAIN SENSOR PROCESSING LOOP
# ==========================================
outdoor_file = '/content/drive/MyDrive/Data/MintsData/Indoor/001e064a1520/combinedValo01Data.csv'
outdoor_data = pd.read_csv(outdoor_file, low_memory=False)
outdoor_data['Datetime'] = pd.to_datetime(outdoor_data['Datetime'], errors='coerce')
outdoor_data = outdoor_data.dropna(subset=['Datetime'])

for indoor_name, indoor_id in sensor_table:
    print(f"\n--- Processing {indoor_name} ({indoor_id}) ---")
    indoor_file = f'/content/drive/MyDrive/Data/MintsData/Indoor/{indoor_id}/{indoor_id}_combined.csv'

    if not os.path.exists(indoor_file):
        print(f"File not found: {indoor_file}. Skipping.")
        continue

    indoor_data = pd.read_csv(indoor_file, low_memory=False)
    # Coerce errors to NaT, then drop rows with NaT in Datetime
    indoor_data['Datetime'] = pd.to_datetime(indoor_data['Datetime'], errors='coerce')
    indoor_data = indoor_data.dropna(subset=['Datetime'])

    # Filter PM values <= 200
    mask = (indoor_data['pm1_0'] <= 200) & (indoor_data['pm2_5'] <= 200) & (indoor_data['pm10_0'] <= 200)
    indoor_data = indoor_data[mask]

    # Temporarily join on exact Datetime to find lag
    temp_join = pd.merge(outdoor_data[['Datetime', 'pm2_5']],
                         indoor_data[['Datetime', 'pm2_5']],
                         on='Datetime', how='inner', suffixes=('_left', '_right')).dropna()

    if temp_join.empty:
        print(f"No overlapping data for {indoor_name}. Skipping.")
        continue

    # Calculate lag
    best_lag = get_best_lag(temp_join['pm2_5_left'], temp_join['pm2_5_right'])
    lag_minutes = best_lag * 10

    # Apply lag shift to indoor data
    indoor_data['Datetime'] = indoor_data['Datetime'] + pd.Timedelta(minutes=lag_minutes)

    # Final Join
    # Final Join (Force MATLAB-style suffixes for overlapping columns)
    joined_data = pd.merge(outdoor_data, indoor_data, on='Datetime', how='inner', suffixes=('_outdoorData', '_indoorData')).dropna()
    print(f"Data aligned and joined successfully for {indoor_name}. Rows: {len(joined_data)}")

    # Rename Columns
    joined_data.rename(columns=rename_dict, inplace=True)

    # Calculate Solar Zenith Angle using pvlib
    times_utc = joined_data['Datetime'].dt.tz_localize('UTC')
    times_local = times_utc.dt.tz_convert('Etc/GMT+6')
    solpos = pvlib.solarposition.get_solarposition(times_local, lat, lon)
    joined_data['SolarZenithAngle'] = solpos['zenith'].values

    # Calculate Target
    joined_data['PM2_5_diff'] = joined_data['Outdoor PM(2.5)'] - joined_data['Indoor PM(2.5)']

    # Filter to predictors and drop any remaining NaNs
    analysis_data = joined_data[['Datetime', 'PM2_5_diff'] + predictor_names].dropna()

    X = analysis_data[predictor_names]
    y = analysis_data['PM2_5_diff']
    datetimes = analysis_data['Datetime']
    all_X_list.append(X)

    # Train/Test Split (80/20)
    X_train, X_test, y_train, y_test, dt_train, dt_test = train_test_split(
        X, y, datetimes, test_size=0.2, random_state=1
    )

    # ==========================================
    # 3. TRAIN GPU XGBOOST MODEL
    # ==========================================
    model = xgb.XGBRegressor(
        n_estimators=100,
        tree_method='hist',
        device='cuda', # Forces execution on the H100/T4 GPU
        random_state=1
    )
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)

    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)

    # --- Plot 1: Actual vs Predicted ---
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_test, color='blue', alpha=0.5, label=f'Test Data (R² = {r2_test:.3f})')
    plt.scatter(y_train, y_pred_train, color='green', alpha=0.2, marker='.', label=f'Train Data (R² = {r2_train:.3f})')

    min_val = min(y.min(), min(y_pred_test.min(), y_pred_train.min()))
    max_val = max(y.max(), max(y_pred_test.max(), y_pred_train.max()))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')

    plt.xlabel('Actual PM2.5 difference (μg/m³)')
    plt.ylabel('Predicted PM2.5 difference (μg/m³)')
    plt.title(f'{indoor_name}: PM2.5 Diff Prediction (included)')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Data/Results/{indoor_name}_PM2_5_outdoor_est_included.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'/content/drive/MyDrive/Data/Results/{indoor_name}_PM2_5_outdoor_est_included.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 4. PERMUTATION IMPORTANCE
    # ==========================================
    print("Calculating Feature Importance...")
    perm_importance = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=1)
    imp_means = perm_importance.importances_mean

    # Sort for plotting
    sorted_idx = imp_means.argsort()

    # --- Plot 2: Feature Importance ---
    plt.figure(figsize=(12, 10))
    plt.barh(np.array(predictor_names)[sorted_idx], imp_means[sorted_idx], color='steelblue')
    plt.xlabel('Mean Importance')
    plt.title(f'{indoor_name}: Feature Importance for PM2.5 Diff (included)')
    plt.grid(axis='x')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Data/Results/{indoor_name}_PM2_5_outdoor_imp.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'/content/drive/MyDrive/Data/Results/{indoor_name}_PM2_5_outdoor_imp.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Save importance stats to list
    imp_dict = {'Sensor': indoor_name, 'Lag_Minutes': lag_minutes,
                'MSE_Test': mse_test, 'RMSE_Test': rmse_test, 'R2_Test': r2_test,
                'MSE_Train': mse_train, 'RMSE_Train': rmse_train, 'R2_Train': r2_train}
    for feat, imp in zip(predictor_names, imp_means):
        imp_dict[feat] = imp
    all_importance_list.append(imp_dict)

    # ==========================================
    # 5. SHAP VALUE CALCULATION (GPU Accelerated)
    # ==========================================
    print("Calculating SHAP values...")
    # Stratified Sampling for Swarm Plot
    test_df = X_test.copy()
    test_df['target'] = y_test
    test_df['datetime'] = dt_test

    # Add minor noise to target to prevent duplicate bin edges in qcut
    target_noise = test_df['target'] + np.random.normal(0, 1e-6, len(test_df))
    test_df['strata'] = pd.qcut(target_noise, q=shap_target_bins, labels=False, duplicates='drop')

    # Sample proportionally from bins
    n_per_bin = max(1, shap_swarm_points // shap_target_bins)
    shap_sample = test_df.groupby('strata', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), n_per_bin), random_state=1),
        include_groups=False
    )

    X_shap = shap_sample[predictor_names]
    y_shap = shap_sample['target'].values
    dt_shap = shap_sample['datetime'].values
    y_shap_pred = model.predict(X_shap)

    # Background KMeans
    kmeans = KMeans(n_clusters=min(shap_background_centroids, len(X_train)), random_state=1, n_init='auto')
    kmeans.fit(X_train)
    background_data = pd.DataFrame(kmeans.cluster_centers_, columns=predictor_names)

    # Execute SHAP
    explainer = shap.TreeExplainer(model, data=background_data)
    shap_values = explainer(X_shap)

    # --- Plot 3: SHAP Summary Swarm Plot ---
    plt.figure(figsize=(12, 10))
    shap.plots.beeswarm(shap_values, max_display=len(predictor_names), show=False)
    plt.title(f'{indoor_name}: SHAP Summary for PM2.5 Diff (included)')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Data/Results/{indoor_name}_PM2_5_outdoor_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'/content/drive/MyDrive/Data/Results/{indoor_name}_PM2_5_outdoor_shap_summary.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Save Mean Absolute SHAP Summary
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    shap_summary_dict = {'Sensor': indoor_name, 'Lag_Minutes': lag_minutes, 'SHAP_Samples': len(X_shap),
                         'MSE_Test': mse_test, 'RMSE_Test': rmse_test, 'R2_Test': r2_test,
                         'MSE_Train': mse_train, 'RMSE_Train': rmse_train, 'R2_Train': r2_train}
    for feat, mean_shap in zip(predictor_names, mean_abs_shap):
        shap_summary_dict[feat] = mean_shap
    all_shap_summary_list.append(shap_summary_dict)

    # Save Long-Format Raw SHAP Values Table
    num_shap_rows = len(X_shap)
    num_predictors = len(predictor_names)

    # Flattening data to match MATLAB's long-format table
    sensor_col = np.repeat(indoor_name, num_shap_rows * num_predictors)
    lag_col = np.repeat(lag_minutes, num_shap_rows * num_predictors)
    query_point_col = np.tile(np.arange(1, num_shap_rows + 1), num_predictors)
    datetime_col = np.tile(dt_shap, num_predictors)
    actual_col = np.tile(y_shap, num_predictors)
    pred_col = np.tile(y_shap_pred, num_predictors)
    predictor_col = np.repeat(predictor_names, num_shap_rows)
    feature_value_col = X_shap.values.T.flatten()
    shap_value_col = shap_values.values.T.flatten()

    long_format_df = pd.DataFrame({
        'Sensor': sensor_col,
        'Lag_Minutes': lag_col,
        'QueryPointIndex': query_point_col,
        'Datetime': datetime_col,
        'Actual_PM2_5_diff': actual_col,
        'Predicted_PM2_5_diff': pred_col,
        'Predictor': predictor_col,
        'FeatureValue': feature_value_col,
        'ShapValue': shap_value_col
    })
    all_shap_values_list.append(long_format_df)

# ==========================================
# 6. EXPORT ALL AGGREGATED TABLES & HEATMAPS
# ==========================================
print("\nExporting aggregated CSV files...")
if all_importance_list:
    df_imp = pd.DataFrame(all_importance_list)
    df_shap = pd.DataFrame(all_shap_summary_list)

    df_imp.to_csv('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_importance_included_GPU.csv', index=False)
    df_shap.to_csv('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_shap_meanabs_included_GPU.csv', index=False)
    pd.concat(all_shap_values_list, ignore_index=True).to_csv('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_shap_values_included_GPU.csv', index=False)

    # ------------------------------------------
    # Correlation Heatmap
    # ------------------------------------------
    if all_X_list:
        print("Generating Correlation Heatmap...")
        X_all = pd.concat(all_X_list, ignore_index=True)
        corr_matrix = X_all.corr()
        corr_matrix.to_csv('/content/drive/MyDrive/Data/Results/predictor_correlation_matrix.csv')

        plt.figure(figsize=(16, 14))
        sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, vmin=-1, vmax=1)
        plt.title('Predictor Correlation Heatmap')
        plt.tight_layout()
        plt.savefig('/content/drive/MyDrive/Data/Results/predictor_correlation_heatmap.png', dpi=300, bbox_inches='tight')
        plt.savefig('/content/drive/MyDrive/Data/Results/predictor_correlation_heatmap.pdf', dpi=300, bbox_inches='tight')
        plt.close()

    # ------------------------------------------
    # Normalized Importance Heatmaps
    # ------------------------------------------
    print("Generating Normalized Importance Heatmaps...")

    # Permutation Heatmap
    df_imp_features = df_imp.set_index('Sensor')[predictor_names]
    df_imp_norm = df_imp_features.div(df_imp_features.max(axis=1), axis=0)
    df_imp_norm.to_csv('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_importance_normalized.csv')

    plt.figure(figsize=(16, 10))
    sns.heatmap(df_imp_norm, cmap='viridis', annot=False)
    plt.title('Normalized Permutation Importance Across Sensors')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_importance_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_importance_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # SHAP Heatmap
    df_shap_features = df_shap.set_index('Sensor')[predictor_names]
    df_shap_norm = df_shap_features.div(df_shap_features.max(axis=1), axis=0)
    df_shap_norm.to_csv('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_shap_normalized.csv')

    plt.figure(figsize=(16, 10))
    sns.heatmap(df_shap_norm, cmap='viridis', annot=False)
    plt.title('Normalized SHAP Importance Across Sensors')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_shap_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig('/content/drive/MyDrive/Data/Results/allSensor_PM2_5_outdoor_shap_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

end_time = time.time()
print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")


--- Processing ind01 (70b3d540f40ce423) ---
Data aligned and joined successfully for ind01. Rows: 48353
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind02 (70b3d540f40ce429) ---
Data aligned and joined successfully for ind02. Rows: 38710
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind03 (70b3d540f40ce435) ---
Data aligned and joined successfully for ind03. Rows: 45573
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind04 (70b3d540f40ce430) ---
Data aligned and joined successfully for ind04. Rows: 41111
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind06 (70b3d540f40ce43c) ---
Data aligned and joined successfully for ind06. Rows: 51137
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind07 (70b3d540f40ce420) ---
Data aligned and joined successfully for ind07. Rows: 42573
Calculating Feature Importance...
Calculating SHAP values...

---

In [ ]:
# ==========================================
# PM 2.5 All Parameters + top10
# ==========================================
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import pvlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
start_time = time.time()
np.random.seed(1)

# Ensure the NEW output directory exists for PM2.5
output_dir = '/content/drive/MyDrive/Data/Results/PM2.5/top10_PM2.5'
os.makedirs(output_dir, exist_ok=True)

# Full Sensor Table
sensor_table = [
    ('ind01', '70b3d540f40ce423'),
    ('ind02', '70b3d540f40ce429'),
    ('ind03', '70b3d540f40ce435'),
    ('ind04', '70b3d540f40ce430'),
    ('ind06', '70b3d540f40ce43c'),
    ('ind07', '70b3d540f40ce420'),
    ('ind08', '70b3d540f40ce425'),
    ('ind09', '70b3d540f40ce438'),
    ('ind10', '70b3d540f40ce436'),
    ('ind11', '70b3d540f40ce433'),
    ('ind12', '70b3d540f40ce43a'),
    ('ind13', '70b3d540f40ce422'),
    ('ind14', '70b3d540f40ce421'),
    ('ind15', '70b3d540f40ce427'),
    ('ind16', '70b3d540f40ce426'),
    ('ind17', '70b3d540f40ce42f'),
    ('ind18', '70b3d540f40ce434'),
    ('ind19', '70b3d540f40ce424'),
    ('ind20', '70b3d540f40ce42d'),
    ('ind21', '70b3d540f40ce43b'),
]

shap_swarm_points = 1000
shap_target_bins = 10
shap_background_centroids = 200

lon = 96.70287
lat = 32.96735

# Storage for All Model CSV outputs
all_importance_list = []
all_shap_summary_list = []
all_shap_values_list = []
all_X_list = []

# Storage for Top 10 Sub-Model CSV outputs
top10_shap_model_imp_list = []
top10_shap_model_summary_list = []

top10_perm_model_imp_list = []
top10_perm_model_summary_list = []

# Storage for Common Features Consensus
common_features_list = []

def get_best_lag(x_series, y_series):
    x = (x_series - x_series.mean()) / x_series.std()
    y = (y_series - y_series.mean()) / y_series.std()
    correlation = signal.correlate(x, y, mode='full')
    lags = signal.correlation_lags(len(x), len(y), mode='full')
    return lags[np.argmax(correlation)]

old_names = [
    'pc0_1','pc0_3','pc0_5', 'pc1_0','pc2_5','pc5_0', 'pc10_0','pm0_1', 'pm0_3', 'pm0_5',
    'pm1_0_outdoorData','pm2_5_outdoorData', 'pm5_0','pm10_0_outdoorData',
    'barrometricPressureBars','airTemperature', 'relativeHumidity','dewPoint',
    'windDirectionMagnetic','windSpeedMetersPerSecond',
    'channelA410nm', 'channelA435nm', 'channelA460nm', 'channelA485nm',
    'channelA510nm', 'channelA535nm', 'channelA560nm', 'channelA585nm',
    'channelA610nm', 'channelA645nm', 'channelA680nm', 'channelA705nm','channelA730nm',
    'channelA760nm', 'channelA810nm', 'channelA860nm', 'channelA900nm', 'channelA940nm',
    'temperature','humidity', 'co2','luminosity','pressure','pm1_0_indoorData',
    'pm2_5_indoorData','pm10_0_indoorData'
]
new_names = [
    'Outdoor PC(0.1)', 'Outdoor PC(0.3)', 'Outdoor PC(0.5)', 'Outdoor PC(1.0)', 'Outdoor PC(2.5)', 'Outdoor PC(5.0)', 'Outdoor PC(10.0)',
    'Outdoor PM(0.1)', 'Outdoor PM(0.3)', 'Outdoor PM(0.5)', 'Outdoor PM(1.0)', 'Outdoor PM(2.5)', 'Outdoor PM(5.0)', 'Outdoor PM(10.0)',
    'Outdoor Pressure', 'Outdoor Temperature', 'Outdoor Humidity', 'Outdoor Dew Point', 'Outdoor Wind Direction', 'Outdoor Wind Speed',
    '410nm', '435nm', '460nm', '485nm', '510nm', '535nm', '560nm', '585nm',
    '610nm', '645nm', '680nm', '705nm', '730nm', '760nm', '810nm', '860nm', '900nm', '940nm',
    'Indoor Temperature', 'Indoor Humidity', 'Indoor CO2', 'Indoor Luminosity', 'Indoor Pressure',
    'Indoor PM(1.0)', 'Indoor PM(2.5)', 'Indoor PM(10.0)'
]
rename_dict = dict(zip(old_names, new_names))

predictor_names = [
    "Outdoor PC(0.1)", "Outdoor PC(0.3)", "Outdoor PC(0.5)", "Outdoor PC(1.0)", "Outdoor PC(2.5)", "Outdoor PC(5.0)", "Outdoor PC(10.0)",
    "Outdoor PM(0.1)", "Outdoor PM(0.3)", "Outdoor PM(0.5)", "Outdoor PM(1.0)", "Outdoor PM(2.5)", "Outdoor PM(5.0)", "Outdoor PM(10.0)",
    "Outdoor Pressure", "Outdoor Temperature", "Outdoor Humidity", "Outdoor Dew Point", "Outdoor Wind Direction", "Outdoor Wind Speed",
    "410nm", "435nm", "460nm", "485nm", "510nm", "535nm", "560nm", "585nm", "610nm", "645nm", "680nm", "705nm",
    "730nm", "760nm", "810nm", "860nm", "900nm", "940nm", "SolarZenithAngle"
]

# ==========================================
# 2. MAIN SENSOR PROCESSING LOOP
# ==========================================
outdoor_file = '/content/drive/MyDrive/Data/MintsData/Indoor/001e064a1520/combinedValo01Data.csv'
outdoor_data = pd.read_csv(outdoor_file, low_memory=False)
outdoor_data['Datetime'] = pd.to_datetime(outdoor_data['Datetime'], errors='coerce')
outdoor_data = outdoor_data.dropna(subset=['Datetime'])

for indoor_name, indoor_id in sensor_table:
    print(f"\n--- Processing {indoor_name} ({indoor_id}) ---")
    indoor_file = f'/content/drive/MyDrive/Data/MintsData/Indoor/{indoor_id}/{indoor_id}_combined.csv'

    if not os.path.exists(indoor_file):
        continue

    indoor_data = pd.read_csv(indoor_file, low_memory=False)
    indoor_data['Datetime'] = pd.to_datetime(indoor_data['Datetime'], errors='coerce')
    indoor_data = indoor_data.dropna(subset=['Datetime'])

    mask = (indoor_data['pm1_0'] <= 200) & (indoor_data['pm2_5'] <= 200) & (indoor_data['pm10_0'] <= 200)
    indoor_data = indoor_data[mask]

    temp_join = pd.merge(outdoor_data[['Datetime', 'pm2_5']], indoor_data[['Datetime', 'pm2_5']],
                         on='Datetime', how='inner', suffixes=('_left', '_right')).dropna()

    if temp_join.empty: continue

    lag_minutes = get_best_lag(temp_join['pm2_5_left'], temp_join['pm2_5_right']) * 10
    indoor_data['Datetime'] = indoor_data['Datetime'] + pd.Timedelta(minutes=lag_minutes)

    joined_data = pd.merge(outdoor_data, indoor_data, on='Datetime', how='inner', suffixes=('_outdoorData', '_indoorData')).dropna()
    joined_data.rename(columns=rename_dict, inplace=True)

    times_utc = joined_data['Datetime'].dt.tz_localize('UTC')
    times_local = times_utc.dt.tz_convert('Etc/GMT+6')
    solpos = pvlib.solarposition.get_solarposition(times_local, lat, lon)
    joined_data['SolarZenithAngle'] = solpos['zenith'].values

    joined_data['PM2_5_diff'] = joined_data['Outdoor PM(2.5)'] - joined_data['Indoor PM(2.5)']
    analysis_data = joined_data[['Datetime', 'PM2_5_diff'] + predictor_names].dropna()

    X = analysis_data[predictor_names]
    y = analysis_data['PM2_5_diff']
    datetimes = analysis_data['Datetime']
    all_X_list.append(X)

    X_train, X_test, y_train, y_test, dt_train, dt_test = train_test_split(X, y, datetimes, test_size=0.2, random_state=1)

    # ==========================================
    # 3. BASE MODEL (ALL FEATURES)
    # ==========================================
    model_all = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1)
    model_all.fit(X_train, y_train)

    y_pred_test_all = model_all.predict(X_test)
    y_pred_train_all = model_all.predict(X_train)
    r2_test_all = r2_score(y_test, y_pred_test_all)
    r2_train_all = r2_score(y_train, y_pred_train_all)

    # Base Plot 1: Actual vs Predicted (All Features)
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_test_all, color='blue', alpha=0.5, label=f'Test Data (R² = {r2_test_all:.3f})')
    plt.scatter(y_train, y_pred_train_all, color='green', alpha=0.2, marker='.', label=f'Train Data (R² = {r2_train_all:.3f})')
    min_val, max_val = min(y.min(), min(y_pred_test_all)), max(y.max(), max(y_pred_test_all))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')
    plt.xlabel('Actual PM2.5 diff (μg/m³)'); plt.ylabel('Predicted PM2.5 diff (μg/m³)')
    plt.title(f'{indoor_name}: PM2.5 Diff Prediction (All Params)')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    plt.grid(True); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_all_est.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Base Permutation Importance
    perm_imp_all = permutation_importance(model_all, X_test, y_test, n_repeats=5, random_state=1).importances_mean
    all_sorted_idx = perm_imp_all.argsort()

    # Base Plot 2: Full Permutation Imp
    plt.figure(figsize=(12, 10))
    plt.barh(np.array(predictor_names)[all_sorted_idx], perm_imp_all[all_sorted_idx], color='steelblue')
    plt.xlabel('Mean Importance'); plt.title(f'{indoor_name}: Feature Imp PM2.5 (All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_all_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Base SHAP
    test_df = X_test.copy(); test_df['target'] = y_test; test_df['datetime'] = dt_test
    test_df['strata'] = pd.qcut(test_df['target'] + np.random.normal(0, 1e-6, len(test_df)), q=shap_target_bins, labels=False, duplicates='drop')
    shap_sample = test_df.groupby('strata', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(1, shap_swarm_points // shap_target_bins)), random_state=1), include_groups=False)
    X_shap = shap_sample[predictor_names]; y_shap = shap_sample['target'].values; dt_shap = shap_sample['datetime'].values

    kmeans_all = KMeans(n_clusters=min(shap_background_centroids, len(X_train)), random_state=1, n_init='auto').fit(X_train)
    expl_all = shap.TreeExplainer(model_all, data=pd.DataFrame(kmeans_all.cluster_centers_, columns=predictor_names))
    shap_vals_all = expl_all(X_shap)
    mean_abs_shap_all = np.abs(shap_vals_all.values).mean(axis=0)

    # Base Plot 3: Full SHAP Summary
    plt.figure(figsize=(12, 10))
    shap.plots.beeswarm(shap_vals_all, max_display=len(predictor_names), show=False)
    plt.title(f'{indoor_name}: SHAP Summary PM2.5 (All Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_all_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 4. EXTRACT TOP 10s & COMMON FEATURES
    # ==========================================
    # Top 10 Permutation (from All)
    top10_perm_idx = all_sorted_idx[-10:]
    top10_perm_features = np.array(predictor_names)[top10_perm_idx].tolist()

    plt.figure(figsize=(10, 6))
    plt.barh(top10_perm_features, perm_imp_all[top10_perm_idx], color='purple')
    plt.xlabel('Mean Importance'); plt.title(f'{indoor_name}: Top 10 Permutation Importance (Extracted from All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_all_model_top10_perm.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Top 10 SHAP (from All)
    shap_sorted_idx = mean_abs_shap_all.argsort()
    top10_shap_idx = shap_sorted_idx[-10:]
    top10_shap_features = np.array(predictor_names)[top10_shap_idx].tolist()

    plt.figure(figsize=(10, 6))
    plt.barh(top10_shap_features, mean_abs_shap_all[top10_shap_idx], color='darkorange')
    plt.xlabel('Mean Absolute SHAP Value'); plt.title(f'{indoor_name}: Top 10 SHAP Importance (Extracted from All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_all_model_top10_shap.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- IDENTIFY COMMON FEATURES (CONSENSUS) ---
    common_features = list(set(top10_perm_features).intersection(set(top10_shap_features)))

    if common_features:
        # Extract indices and values for the common features
        common_idx = [predictor_names.index(f) for f in common_features]
        common_perm_vals = perm_imp_all[common_idx]
        common_shap_vals = mean_abs_shap_all[common_idx]

        # Sort by SHAP value to make the side-by-side plot flow visually
        sort_order = np.argsort(common_shap_vals)
        c_feat_sorted = np.array(common_features)[sort_order]
        c_perm_sorted = np.array(common_perm_vals)[sort_order]
        c_shap_sorted = np.array(common_shap_vals)[sort_order]

        # Side-by-Side Highlight Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

        ax1.barh(c_feat_sorted, c_perm_sorted, color='purple')
        ax1.set_xlabel('Mean Permutation Importance')
        ax1.set_title('Permutation Scoring')
        ax1.grid(axis='x')

        ax2.barh(c_feat_sorted, c_shap_sorted, color='darkorange')
        ax2.set_xlabel('Mean Absolute SHAP Value')
        ax2.set_title('SHAP Scoring')
        ax2.grid(axis='x')

        fig.suptitle(f'{indoor_name}: Consensus Top Features (Present in both SHAP & Permutation)', fontsize=16)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_common_top_features.png', dpi=300, bbox_inches='tight')
        plt.close()

        # Save to consensus list for CSV output
        for f, p_val, s_val in zip(c_feat_sorted, c_perm_sorted, c_shap_sorted):
            common_features_list.append({
                'Sensor': indoor_name,
                'Feature': f,
                'Permutation_Importance': p_val,
                'SHAP_Value': s_val
            })

    # ==========================================
    # 5. TRAIN NEW SUB-MODELS (TOP 10 ONLY)
    # ==========================================
    # --- MODEL 2: SHAP TOP 10 ---
    X_train_shap10, X_test_shap10 = X_train[top10_shap_features], X_test[top10_shap_features]
    model_shap10 = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1).fit(X_train_shap10, y_train)
    y_pred_test_shap10 = model_shap10.predict(X_test_shap10)
    r2_test_shap10 = r2_score(y_test, y_pred_test_shap10)

    perm_imp_shap10 = permutation_importance(model_shap10, X_test_shap10, y_test, n_repeats=5, random_state=1).importances_mean

    plt.figure(figsize=(10, 6))
    plt.barh(np.array(top10_shap_features)[perm_imp_shap10.argsort()], perm_imp_shap10[perm_imp_shap10.argsort()], color='darkorange')
    plt.title(f'{indoor_name}: Feature Imp (Model trained ON Top 10 SHAP Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_shap10_model_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    bg_shap10 = pd.DataFrame(KMeans(n_clusters=min(shap_background_centroids, len(X_train_shap10)), random_state=1, n_init='auto').fit(X_train_shap10).cluster_centers_, columns=top10_shap_features)
    expl_shap10 = shap.TreeExplainer(model_shap10, data=bg_shap10)
    sv_shap10 = expl_shap10(X_shap[top10_shap_features])

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(sv_shap10, max_display=10, show=False)
    plt.title(f'{indoor_name}: SHAP Summary (Model trained ON Top 10 SHAP Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_shap10_model_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- MODEL 3: PERM TOP 10 ---
    X_train_perm10, X_test_perm10 = X_train[top10_perm_features], X_test[top10_perm_features]
    model_perm10 = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1).fit(X_train_perm10, y_train)
    y_pred_test_perm10 = model_perm10.predict(X_test_perm10)
    r2_test_perm10 = r2_score(y_test, y_pred_test_perm10)

    perm_imp_perm10 = permutation_importance(model_perm10, X_test_perm10, y_test, n_repeats=5, random_state=1).importances_mean

    plt.figure(figsize=(10, 6))
    plt.barh(np.array(top10_perm_features)[perm_imp_perm10.argsort()], perm_imp_perm10[perm_imp_perm10.argsort()], color='purple')
    plt.title(f'{indoor_name}: Feature Imp (Model trained ON Top 10 Perm Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_perm10_model_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    bg_perm10 = pd.DataFrame(KMeans(n_clusters=min(shap_background_centroids, len(X_train_perm10)), random_state=1, n_init='auto').fit(X_train_perm10).cluster_centers_, columns=top10_perm_features)
    expl_perm10 = shap.TreeExplainer(model_perm10, data=bg_perm10)
    sv_perm10 = expl_perm10(X_shap[top10_perm_features])

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(sv_perm10, max_display=10, show=False)
    plt.title(f'{indoor_name}: SHAP Summary (Model trained ON Top 10 Perm Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_perm10_model_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 6. OVERLAY PLOT (ALL vs SHAP10 vs PERM10)
    # ==========================================
    plt.figure(figsize=(12, 10))
    # Base All-Param Data
    plt.scatter(y_train, y_pred_train_all, color='green', alpha=0.1, marker='.', label=f'Train All Params (R² = {r2_train_all:.3f})')
    plt.scatter(y_test, y_pred_test_all, color='blue', alpha=0.3, label=f'Test All Params (R² = {r2_test_all:.3f})')
    # Top 10 SHAP Model Data
    plt.scatter(y_test, y_pred_test_shap10, color='darkorange', alpha=0.5, marker='x', label=f'Test Top 10 SHAP Model (R² = {r2_test_shap10:.3f})')
    # Top 10 Perm Model Data
    plt.scatter(y_test, y_pred_test_perm10, color='purple', alpha=0.5, marker='+', label=f'Test Top 10 Perm Model (R² = {r2_test_perm10:.3f})')

    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')
    plt.xlabel('Actual PM2.5 diff (μg/m³)'); plt.ylabel('Predicted PM2.5 diff (μg/m³)')
    plt.title(f'{indoor_name}: Overlay PM2.5 Prediction Comparison')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=2)
    plt.grid(True); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM2_5_overlay_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 7. POPULATE DICTIONARIES FOR CSV EXPORTS
    # ==========================================
    all_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_all}; all_imp_dict.update(zip(predictor_names, perm_imp_all)); all_importance_list.append(all_imp_dict)
    all_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_all}; all_shap_dict.update(zip(predictor_names, mean_abs_shap_all)); all_shap_summary_list.append(all_shap_dict)

    s10_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_shap10}; s10_imp_dict.update(zip(top10_shap_features, perm_imp_shap10)); top10_shap_model_imp_list.append(s10_imp_dict)
    s10_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_shap10}; s10_shap_dict.update(zip(top10_shap_features, np.abs(sv_shap10.values).mean(axis=0))); top10_shap_model_summary_list.append(s10_shap_dict)

    p10_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_perm10}; p10_imp_dict.update(zip(top10_perm_features, perm_imp_perm10)); top10_perm_model_imp_list.append(p10_imp_dict)
    p10_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_perm10}; p10_shap_dict.update(zip(top10_perm_features, np.abs(sv_perm10.values).mean(axis=0))); top10_perm_model_summary_list.append(p10_shap_dict)

# ==========================================
# 8. EXPORT ALL AGGREGATED TABLES
# ==========================================
print("\nExporting aggregated CSV files...")

if all_importance_list:
    # Original Base Outputs
    pd.DataFrame(all_importance_list).to_csv(f'{output_dir}/allSensor_PM2_5_all_model_imp.csv', index=False)
    pd.DataFrame(all_shap_summary_list).to_csv(f'{output_dir}/allSensor_PM2_5_all_model_shap_meanabs.csv', index=False)

    # New Top 10 Sub-Model Outputs
    pd.DataFrame(top10_shap_model_imp_list).to_csv(f'{output_dir}/allSensor_PM2_5_shap10_model_imp.csv', index=False)
    pd.DataFrame(top10_shap_model_summary_list).to_csv(f'{output_dir}/allSensor_PM2_5_shap10_model_shap_meanabs.csv', index=False)

    pd.DataFrame(top10_perm_model_imp_list).to_csv(f'{output_dir}/allSensor_PM2_5_perm10_model_imp.csv', index=False)
    pd.DataFrame(top10_perm_model_summary_list).to_csv(f'{output_dir}/allSensor_PM2_5_perm10_model_shap_meanabs.csv', index=False)

    # Consensus Features Output
    if common_features_list:
        pd.DataFrame(common_features_list).to_csv(f'{output_dir}/allSensor_PM2_5_common_consensus_features.csv', index=False)

end_time = time.time()
print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")


--- Processing ind01 (70b3d540f40ce423) ---

--- Processing ind02 (70b3d540f40ce429) ---

--- Processing ind03 (70b3d540f40ce435) ---

--- Processing ind04 (70b3d540f40ce430) ---

--- Processing ind06 (70b3d540f40ce43c) ---

--- Processing ind07 (70b3d540f40ce420) ---

--- Processing ind08 (70b3d540f40ce425) ---

--- Processing ind09 (70b3d540f40ce438) ---

--- Processing ind10 (70b3d540f40ce436) ---

--- Processing ind11 (70b3d540f40ce433) ---

--- Processing ind12 (70b3d540f40ce43a) ---

--- Processing ind13 (70b3d540f40ce422) ---

--- Processing ind14 (70b3d540f40ce421) ---

--- Processing ind15 (70b3d540f40ce427) ---

--- Processing ind16 (70b3d540f40ce426) ---

--- Processing ind17 (70b3d540f40ce42f) ---

--- Processing ind18 (70b3d540f40ce434) ---

--- Processing ind19 (70b3d540f40ce424) ---

--- Processing ind20 (70b3d540f40ce42d) ---

--- Processing ind21 (70b3d540f40ce43b) ---

Exporting aggregated CSV files...
Total processing time: 3.86 minutes.


In [ ]:
# ==========================================
# PM 1.0 All Parameters
# ==========================================
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import pvlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
start_time = time.time()
np.random.seed(1)

# Full Sensor Table from MATLAB
sensor_table = [
    ('ind01', '70b3d540f40ce423'),
    ('ind02', '70b3d540f40ce429'),
    ('ind03', '70b3d540f40ce435'),
    ('ind04', '70b3d540f40ce430'),
    ('ind06', '70b3d540f40ce43c'),
    ('ind07', '70b3d540f40ce420'),
    ('ind08', '70b3d540f40ce425'),
    ('ind09', '70b3d540f40ce438'),
    ('ind10', '70b3d540f40ce436'),
    ('ind11', '70b3d540f40ce433'),
    ('ind12', '70b3d540f40ce43a'),
    ('ind13', '70b3d540f40ce422'),
    ('ind14', '70b3d540f40ce421'),
    ('ind15', '70b3d540f40ce427'),
    ('ind16', '70b3d540f40ce426'),
    ('ind17', '70b3d540f40ce42f'),
    ('ind18', '70b3d540f40ce434'),
    ('ind19', '70b3d540f40ce424'),
    ('ind20', '70b3d540f40ce42d'),
    ('ind21', '70b3d540f40ce43b'),
]

# Runtime settings
shap_swarm_points = 1000
shap_target_bins = 10
shap_background_centroids = 200

# Geographic Coordinates for Solar Angle
lon = 96.70287
lat = 32.96735

# Storage for final CSV outputs
all_importance_list = []
all_shap_summary_list = []
all_shap_values_list = []
all_X_list = []

# Helper function to compute optimal lag via cross-correlation
def get_best_lag(x_series, y_series):
    x = (x_series - x_series.mean()) / x_series.std()
    y = (y_series - y_series.mean()) / y_series.std()
    correlation = signal.correlate(x, y, mode='full')
    lags = signal.correlation_lags(len(x), len(y), mode='full')
    best_lag = lags[np.argmax(correlation)]
    return best_lag

# Column renaming mappings
old_names = [
    'pc0_1','pc0_3','pc0_5', 'pc1_0','pc2_5','pc5_0', 'pc10_0','pm0_1', 'pm0_3', 'pm0_5',
    'pm1_0_outdoorData','pm2_5_outdoorData', 'pm5_0','pm10_0_outdoorData',
    'barrometricPressureBars','airTemperature', 'relativeHumidity','dewPoint',
    'windDirectionMagnetic','windSpeedMetersPerSecond',
    'channelA410nm', 'channelA435nm', 'channelA460nm', 'channelA485nm',
    'channelA510nm', 'channelA535nm', 'channelA560nm', 'channelA585nm',
    'channelA610nm', 'channelA645nm', 'channelA680nm', 'channelA705nm','channelA730nm',
    'channelA760nm', 'channelA810nm', 'channelA860nm', 'channelA900nm', 'channelA940nm',
    'temperature','humidity', 'co2','luminosity','pressure','pm1_0_indoorData',
    'pm2_5_indoorData','pm10_0_indoorData'
]
new_names = [
    'Outdoor PC(0.1)', 'Outdoor PC(0.3)', 'Outdoor PC(0.5)', 'Outdoor PC(1.0)', 'Outdoor PC(2.5)', 'Outdoor PC(5.0)', 'Outdoor PC(10.0)',
    'Outdoor PM(0.1)', 'Outdoor PM(0.3)', 'Outdoor PM(0.5)', 'Outdoor PM(1.0)', 'Outdoor PM(2.5)', 'Outdoor PM(5.0)', 'Outdoor PM(10.0)',
    'Outdoor Pressure', 'Outdoor Temperature', 'Outdoor Humidity', 'Outdoor Dew Point', 'Outdoor Wind Direction', 'Outdoor Wind Speed',
    '410nm', '435nm', '460nm', '485nm', '510nm', '535nm', '560nm', '585nm',
    '610nm', '645nm', '680nm', '705nm', '730nm', '760nm', '810nm', '860nm', '900nm', '940nm',
    'Indoor Temperature', 'Indoor Humidity', 'Indoor CO2', 'Indoor Luminosity', 'Indoor Pressure',
    'Indoor PM(1.0)', 'Indoor PM(2.5)', 'Indoor PM(10.0)'
]
rename_dict = dict(zip(old_names, new_names))

predictor_names = [
    "Outdoor PC(0.1)", "Outdoor PC(0.3)", "Outdoor PC(0.5)", "Outdoor PC(1.0)", "Outdoor PC(2.5)", "Outdoor PC(5.0)", "Outdoor PC(10.0)",
    "Outdoor PM(0.1)", "Outdoor PM(0.3)", "Outdoor PM(0.5)", "Outdoor PM(1.0)", "Outdoor PM(2.5)", "Outdoor PM(5.0)", "Outdoor PM(10.0)",
    "Outdoor Pressure", "Outdoor Temperature", "Outdoor Humidity", "Outdoor Dew Point", "Outdoor Wind Direction", "Outdoor Wind Speed",
    "410nm", "435nm", "460nm", "485nm", "510nm", "535nm", "560nm", "585nm", "610nm", "645nm", "680nm", "705nm",
    "730nm", "760nm", "810nm", "860nm", "900nm", "940nm", "SolarZenithAngle"
]

# ==========================================
# 2. MAIN SENSOR PROCESSING LOOP
# ==========================================
outdoor_file = '/content/drive/MyDrive/Data/MintsData/Indoor/001e064a1520/combinedValo01Data.csv'
outdoor_data = pd.read_csv(outdoor_file, low_memory=False)
outdoor_data['Datetime'] = pd.to_datetime(outdoor_data['Datetime'], errors='coerce')
outdoor_data = outdoor_data.dropna(subset=['Datetime'])

for indoor_name, indoor_id in sensor_table:
    print(f"\n--- Processing {indoor_name} ({indoor_id}) ---")
    indoor_file = f'/content/drive/MyDrive/Data/MintsData/Indoor/{indoor_id}/{indoor_id}_combined.csv'

    if not os.path.exists(indoor_file):
        print(f"File not found: {indoor_file}. Skipping.")
        continue

    indoor_data = pd.read_csv(indoor_file, low_memory=False)
    # Coerce errors to NaT, then drop rows with NaT in Datetime
    indoor_data['Datetime'] = pd.to_datetime(indoor_data['Datetime'], errors='coerce')
    indoor_data = indoor_data.dropna(subset=['Datetime'])

    # Filter PM values <= 200
    mask = (indoor_data['pm1_0'] <= 200) & (indoor_data['pm2_5'] <= 200) & (indoor_data['pm10_0'] <= 200)
    indoor_data = indoor_data[mask]

    # Temporarily join on exact Datetime to find lag (NOW USING PM1.0)
    temp_join = pd.merge(outdoor_data[['Datetime', 'pm1_0']],
                         indoor_data[['Datetime', 'pm1_0']],
                         on='Datetime', how='inner', suffixes=('_left', '_right')).dropna()

    if temp_join.empty:
        print(f"No overlapping data for {indoor_name}. Skipping.")
        continue

    # Calculate lag (NOW USING PM1.0)
    best_lag = get_best_lag(temp_join['pm1_0_left'], temp_join['pm1_0_right'])
    lag_minutes = best_lag * 10

    # Apply lag shift to indoor data
    indoor_data['Datetime'] = indoor_data['Datetime'] + pd.Timedelta(minutes=lag_minutes)

    # Final Join
    # Final Join (Force MATLAB-style suffixes for overlapping columns)
    joined_data = pd.merge(outdoor_data, indoor_data, on='Datetime', how='inner', suffixes=('_outdoorData', '_indoorData')).dropna()
    print(f"Data aligned and joined successfully for {indoor_name}. Rows: {len(joined_data)}")

    # Rename Columns
    joined_data.rename(columns=rename_dict, inplace=True)

    # Calculate Solar Zenith Angle using pvlib
    times_utc = joined_data['Datetime'].dt.tz_localize('UTC')
    times_local = times_utc.dt.tz_convert('Etc/GMT+6')
    solpos = pvlib.solarposition.get_solarposition(times_local, lat, lon)
    joined_data['SolarZenithAngle'] = solpos['zenith'].values

    # Calculate Target (NOW USING PM1.0)
    joined_data['PM1_0_diff'] = joined_data['Outdoor PM(1.0)'] - joined_data['Indoor PM(1.0)']

    # Filter to predictors and drop any remaining NaNs (NOW USING PM1_0_diff)
    analysis_data = joined_data[['Datetime', 'PM1_0_diff'] + predictor_names].dropna()

    X = analysis_data[predictor_names]
    y = analysis_data['PM1_0_diff']
    datetimes = analysis_data['Datetime']
    all_X_list.append(X)

    # Train/Test Split (80/20)
    X_train, X_test, y_train, y_test, dt_train, dt_test = train_test_split(
        X, y, datetimes, test_size=0.2, random_state=1
    )

    # ==========================================
    # 3. TRAIN GPU XGBOOST MODEL
    # ==========================================
    model = xgb.XGBRegressor(
        n_estimators=100,
        tree_method='hist',
        device='cuda', # Forces execution on the H100/T4 GPU
        random_state=1
    )
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)

    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)

    # --- Plot 1: Actual vs Predicted ---
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_test, color='blue', alpha=0.5, label=f'Test Data (R² = {r2_test:.3f})')
    plt.scatter(y_train, y_pred_train, color='green', alpha=0.2, marker='.', label=f'Train Data (R² = {r2_train:.3f})')

    min_val = min(y.min(), min(y_pred_test.min(), y_pred_train.min()))
    max_val = max(y.max(), max(y_pred_test.max(), y_pred_train.max()))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')

    plt.xlabel('Actual PM1.0 difference (μg/m³)')
    plt.ylabel('Predicted PM1.0 difference (μg/m³)')
    plt.title(f'{indoor_name}: PM1.0 Diff Prediction (included)')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Data/Results/PM1.0/{indoor_name}_PM1_0_outdoor_est_included.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'/content/drive/MyDrive/Data/Results/PM1.0/{indoor_name}_PM1_0_outdoor_est_included.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 4. PERMUTATION IMPORTANCE
    # ==========================================
    print("Calculating Feature Importance...")
    perm_importance = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=1)
    imp_means = perm_importance.importances_mean

    # Sort for plotting
    sorted_idx = imp_means.argsort()

    # --- Plot 2: Feature Importance ---
    plt.figure(figsize=(12, 10))
    plt.barh(np.array(predictor_names)[sorted_idx], imp_means[sorted_idx], color='steelblue')
    plt.xlabel('Mean Importance')
    plt.title(f'{indoor_name}: Feature Importance for PM1.0 Diff (included)')
    plt.grid(axis='x')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Data/Results/PM1.0/{indoor_name}_PM1_0_outdoor_imp.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'/content/drive/MyDrive/Data/Results/PM1.0/{indoor_name}_PM1_0_outdoor_imp.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Save importance stats to list
    imp_dict = {'Sensor': indoor_name, 'Lag_Minutes': lag_minutes,
                'MSE_Test': mse_test, 'RMSE_Test': rmse_test, 'R2_Test': r2_test,
                'MSE_Train': mse_train, 'RMSE_Train': rmse_train, 'R2_Train': r2_train}
    for feat, imp in zip(predictor_names, imp_means):
        imp_dict[feat] = imp
    all_importance_list.append(imp_dict)

    # ==========================================
    # 5. SHAP VALUE CALCULATION (GPU Accelerated)
    # ==========================================
    print("Calculating SHAP values...")
    # Stratified Sampling for Swarm Plot
    test_df = X_test.copy()
    test_df['target'] = y_test
    test_df['datetime'] = dt_test

    # Add minor noise to target to prevent duplicate bin edges in qcut
    target_noise = test_df['target'] + np.random.normal(0, 1e-6, len(test_df))
    test_df['strata'] = pd.qcut(target_noise, q=shap_target_bins, labels=False, duplicates='drop')

    # Sample proportionally from bins
    n_per_bin = max(1, shap_swarm_points // shap_target_bins)
    shap_sample = test_df.groupby('strata', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), n_per_bin), random_state=1),
        include_groups=False
    )

    X_shap = shap_sample[predictor_names]
    y_shap = shap_sample['target'].values
    dt_shap = shap_sample['datetime'].values
    y_shap_pred = model.predict(X_shap)

    # Background KMeans
    kmeans = KMeans(n_clusters=min(shap_background_centroids, len(X_train)), random_state=1, n_init='auto')
    kmeans.fit(X_train)
    background_data = pd.DataFrame(kmeans.cluster_centers_, columns=predictor_names)

    # Execute SHAP
    explainer = shap.TreeExplainer(model, data=background_data)
    shap_values = explainer(X_shap)

    # --- Plot 3: SHAP Summary Swarm Plot ---
    plt.figure(figsize=(12, 10))
    shap.plots.beeswarm(shap_values, max_display=len(predictor_names), show=False)
    plt.title(f'{indoor_name}: SHAP Summary for PM1.0 Diff (included)')
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Data/Results/PM1.0/{indoor_name}_PM1_0_outdoor_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'/content/drive/MyDrive/Data/Results/PM1.0/{indoor_name}_PM1_0_outdoor_shap_summary.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Save Mean Absolute SHAP Summary
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    shap_summary_dict = {'Sensor': indoor_name, 'Lag_Minutes': lag_minutes, 'SHAP_Samples': len(X_shap),
                         'MSE_Test': mse_test, 'RMSE_Test': rmse_test, 'R2_Test': r2_test,
                         'MSE_Train': mse_train, 'RMSE_Train': rmse_train, 'R2_Train': r2_train}
    for feat, mean_shap in zip(predictor_names, mean_abs_shap):
        shap_summary_dict[feat] = mean_shap
    all_shap_summary_list.append(shap_summary_dict)

    # Save Long-Format Raw SHAP Values Table
    num_shap_rows = len(X_shap)
    num_predictors = len(predictor_names)

    # Flattening data to match MATLAB's long-format table
    sensor_col = np.repeat(indoor_name, num_shap_rows * num_predictors)
    lag_col = np.repeat(lag_minutes, num_shap_rows * num_predictors)
    query_point_col = np.tile(np.arange(1, num_shap_rows + 1), num_predictors)
    datetime_col = np.tile(dt_shap, num_predictors)
    actual_col = np.tile(y_shap, num_predictors)
    pred_col = np.tile(y_shap_pred, num_predictors)
    predictor_col = np.repeat(predictor_names, num_shap_rows)
    feature_value_col = X_shap.values.T.flatten()
    shap_value_col = shap_values.values.T.flatten()

    long_format_df = pd.DataFrame({
        'Sensor': sensor_col,
        'Lag_Minutes': lag_col,
        'QueryPointIndex': query_point_col,
        'Datetime': datetime_col,
        'Actual_PM1_0_diff': actual_col,
        'Predicted_PM1_0_diff': pred_col,
        'Predictor': predictor_col,
        'FeatureValue': feature_value_col,
        'ShapValue': shap_value_col
    })
    all_shap_values_list.append(long_format_df)

# ==========================================
# 6. EXPORT ALL AGGREGATED TABLES & HEATMAPS
# ==========================================
print("\nExporting aggregated CSV files...")
if all_importance_list:
    df_imp = pd.DataFrame(all_importance_list)
    df_shap = pd.DataFrame(all_shap_summary_list)

    df_imp.to_csv('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_importance_included_GPU.csv', index=False)
    df_shap.to_csv('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_shap_meanabs_included_GPU.csv', index=False)
    pd.concat(all_shap_values_list, ignore_index=True).to_csv('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_shap_values_included_GPU.csv', index=False)

    # ------------------------------------------
    # Correlation Heatmap (Kept identical but output changed to PM1.0 if desired, leaving base name since it's just predictors)
    # ------------------------------------------
    if all_X_list:
        print("Generating Correlation Heatmap...")
        X_all = pd.concat(all_X_list, ignore_index=True)
        corr_matrix = X_all.corr()
        corr_matrix.to_csv('/content/drive/MyDrive/Data/Results/PM1.0/predictor_correlation_matrix.csv')

        plt.figure(figsize=(16, 14))
        sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, vmin=-1, vmax=1)
        plt.title('Predictor Correlation Heatmap')
        plt.tight_layout()
        plt.savefig('/content/drive/MyDrive/Data/Results/PM1.0/predictor_correlation_heatmap.png', dpi=300, bbox_inches='tight')
        plt.savefig('/content/drive/MyDrive/Data/Results/PM1.0/predictor_correlation_heatmap.pdf', dpi=300, bbox_inches='tight')
        plt.close()

    # ------------------------------------------
    # Normalized Importance Heatmaps
    # ------------------------------------------
    print("Generating Normalized Importance Heatmaps...")

    # Permutation Heatmap
    df_imp_features = df_imp.set_index('Sensor')[predictor_names]
    df_imp_norm = df_imp_features.div(df_imp_features.max(axis=1), axis=0)
    df_imp_norm.to_csv('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_importance_normalized.csv')

    plt.figure(figsize=(16, 10))
    sns.heatmap(df_imp_norm, cmap='viridis', annot=False)
    plt.title('Normalized Permutation Importance Across Sensors (PM1.0)')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_importance_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_importance_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # SHAP Heatmap
    df_shap_features = df_shap.set_index('Sensor')[predictor_names]
    df_shap_norm = df_shap_features.div(df_shap_features.max(axis=1), axis=0)
    df_shap_norm.to_csv('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_shap_normalized.csv')

    plt.figure(figsize=(16, 10))
    sns.heatmap(df_shap_norm, cmap='viridis', annot=False)
    plt.title('Normalized SHAP Importance Across Sensors (PM1.0)')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_shap_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig('/content/drive/MyDrive/Data/Results/PM1.0/allSensor_PM1_0_outdoor_shap_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

end_time = time.time()
print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")


--- Processing ind01 (70b3d540f40ce423) ---
Data aligned and joined successfully for ind01. Rows: 48353
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind02 (70b3d540f40ce429) ---
Data aligned and joined successfully for ind02. Rows: 38710
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind03 (70b3d540f40ce435) ---
Data aligned and joined successfully for ind03. Rows: 45576
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind04 (70b3d540f40ce430) ---
Data aligned and joined successfully for ind04. Rows: 41111
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind06 (70b3d540f40ce43c) ---
Data aligned and joined successfully for ind06. Rows: 51130
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind07 (70b3d540f40ce420) ---
Data aligned and joined successfully for ind07. Rows: 42573
Calculating Feature Importance...
Calculating SHAP values...

---

In [ ]:
# ==========================================
# PM 1.0 All Parameters + top10
# ==========================================
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import pvlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
start_time = time.time()
np.random.seed(1)

# Ensure the NEW output directory exists for PM1.0
output_dir = '/content/drive/MyDrive/Data/Results/PM1.0/top10_PM1'
os.makedirs(output_dir, exist_ok=True)

# Full Sensor Table
sensor_table = [
    ('ind01', '70b3d540f40ce423'),
    ('ind02', '70b3d540f40ce429'),
    ('ind03', '70b3d540f40ce435'),
    ('ind04', '70b3d540f40ce430'),
    ('ind06', '70b3d540f40ce43c'),
    ('ind07', '70b3d540f40ce420'),
    ('ind08', '70b3d540f40ce425'),
    ('ind09', '70b3d540f40ce438'),
    ('ind10', '70b3d540f40ce436'),
    ('ind11', '70b3d540f40ce433'),
    ('ind12', '70b3d540f40ce43a'),
    ('ind13', '70b3d540f40ce422'),
    ('ind14', '70b3d540f40ce421'),
    ('ind15', '70b3d540f40ce427'),
    ('ind16', '70b3d540f40ce426'),
    ('ind17', '70b3d540f40ce42f'),
    ('ind18', '70b3d540f40ce434'),
    ('ind19', '70b3d540f40ce424'),
    ('ind20', '70b3d540f40ce42d'),
    ('ind21', '70b3d540f40ce43b'),
]

shap_swarm_points = 1000
shap_target_bins = 10
shap_background_centroids = 200

lon = 96.70287
lat = 32.96735

# Storage for All Model CSV outputs
all_importance_list = []
all_shap_summary_list = []
all_shap_values_list = []
all_X_list = []

# Storage for Top 10 Sub-Model CSV outputs
top10_shap_model_imp_list = []
top10_shap_model_summary_list = []

top10_perm_model_imp_list = []
top10_perm_model_summary_list = []

# Storage for Common Features Consensus
common_features_list = []

def get_best_lag(x_series, y_series):
    x = (x_series - x_series.mean()) / x_series.std()
    y = (y_series - y_series.mean()) / y_series.std()
    correlation = signal.correlate(x, y, mode='full')
    lags = signal.correlation_lags(len(x), len(y), mode='full')
    return lags[np.argmax(correlation)]

old_names = [
    'pc0_1','pc0_3','pc0_5', 'pc1_0','pc2_5','pc5_0', 'pc10_0','pm0_1', 'pm0_3', 'pm0_5',
    'pm1_0_outdoorData','pm2_5_outdoorData', 'pm5_0','pm10_0_outdoorData',
    'barrometricPressureBars','airTemperature', 'relativeHumidity','dewPoint',
    'windDirectionMagnetic','windSpeedMetersPerSecond',
    'channelA410nm', 'channelA435nm', 'channelA460nm', 'channelA485nm',
    'channelA510nm', 'channelA535nm', 'channelA560nm', 'channelA585nm',
    'channelA610nm', 'channelA645nm', 'channelA680nm', 'channelA705nm','channelA730nm',
    'channelA760nm', 'channelA810nm', 'channelA860nm', 'channelA900nm', 'channelA940nm',
    'temperature','humidity', 'co2','luminosity','pressure','pm1_0_indoorData',
    'pm2_5_indoorData','pm10_0_indoorData'
]
new_names = [
    'Outdoor PC(0.1)', 'Outdoor PC(0.3)', 'Outdoor PC(0.5)', 'Outdoor PC(1.0)', 'Outdoor PC(2.5)', 'Outdoor PC(5.0)', 'Outdoor PC(10.0)',
    'Outdoor PM(0.1)', 'Outdoor PM(0.3)', 'Outdoor PM(0.5)', 'Outdoor PM(1.0)', 'Outdoor PM(2.5)', 'Outdoor PM(5.0)', 'Outdoor PM(10.0)',
    'Outdoor Pressure', 'Outdoor Temperature', 'Outdoor Humidity', 'Outdoor Dew Point', 'Outdoor Wind Direction', 'Outdoor Wind Speed',
    '410nm', '435nm', '460nm', '485nm', '510nm', '535nm', '560nm', '585nm',
    '610nm', '645nm', '680nm', '705nm', '730nm', '760nm', '810nm', '860nm', '900nm', '940nm',
    'Indoor Temperature', 'Indoor Humidity', 'Indoor CO2', 'Indoor Luminosity', 'Indoor Pressure',
    'Indoor PM(1.0)', 'Indoor PM(2.5)', 'Indoor PM(10.0)'
]
rename_dict = dict(zip(old_names, new_names))

predictor_names = [
    "Outdoor PC(0.1)", "Outdoor PC(0.3)", "Outdoor PC(0.5)", "Outdoor PC(1.0)", "Outdoor PC(2.5)", "Outdoor PC(5.0)", "Outdoor PC(10.0)",
    "Outdoor PM(0.1)", "Outdoor PM(0.3)", "Outdoor PM(0.5)", "Outdoor PM(1.0)", "Outdoor PM(2.5)", "Outdoor PM(5.0)", "Outdoor PM(10.0)",
    "Outdoor Pressure", "Outdoor Temperature", "Outdoor Humidity", "Outdoor Dew Point", "Outdoor Wind Direction", "Outdoor Wind Speed",
    "410nm", "435nm", "460nm", "485nm", "510nm", "535nm", "560nm", "585nm", "610nm", "645nm", "680nm", "705nm",
    "730nm", "760nm", "810nm", "860nm", "900nm", "940nm", "SolarZenithAngle"
]

# ==========================================
# 2. MAIN SENSOR PROCESSING LOOP
# ==========================================
outdoor_file = '/content/drive/MyDrive/Data/MintsData/Indoor/001e064a1520/combinedValo01Data.csv'
outdoor_data = pd.read_csv(outdoor_file, low_memory=False)
outdoor_data['Datetime'] = pd.to_datetime(outdoor_data['Datetime'], errors='coerce')
outdoor_data = outdoor_data.dropna(subset=['Datetime'])

for indoor_name, indoor_id in sensor_table:
    print(f"\n--- Processing {indoor_name} ({indoor_id}) ---")
    indoor_file = f'/content/drive/MyDrive/Data/MintsData/Indoor/{indoor_id}/{indoor_id}_combined.csv'

    if not os.path.exists(indoor_file):
        continue

    indoor_data = pd.read_csv(indoor_file, low_memory=False)
    indoor_data['Datetime'] = pd.to_datetime(indoor_data['Datetime'], errors='coerce')
    indoor_data = indoor_data.dropna(subset=['Datetime'])

    mask = (indoor_data['pm1_0'] <= 200) & (indoor_data['pm2_5'] <= 200) & (indoor_data['pm10_0'] <= 200)
    indoor_data = indoor_data[mask]

    temp_join = pd.merge(outdoor_data[['Datetime', 'pm1_0']], indoor_data[['Datetime', 'pm1_0']],
                         on='Datetime', how='inner', suffixes=('_left', '_right')).dropna()

    if temp_join.empty: continue

    lag_minutes = get_best_lag(temp_join['pm1_0_left'], temp_join['pm1_0_right']) * 10
    indoor_data['Datetime'] = indoor_data['Datetime'] + pd.Timedelta(minutes=lag_minutes)

    joined_data = pd.merge(outdoor_data, indoor_data, on='Datetime', how='inner', suffixes=('_outdoorData', '_indoorData')).dropna()
    joined_data.rename(columns=rename_dict, inplace=True)

    times_utc = joined_data['Datetime'].dt.tz_localize('UTC')
    times_local = times_utc.dt.tz_convert('Etc/GMT+6')
    solpos = pvlib.solarposition.get_solarposition(times_local, lat, lon)
    joined_data['SolarZenithAngle'] = solpos['zenith'].values

    joined_data['PM1_0_diff'] = joined_data['Outdoor PM(1.0)'] - joined_data['Indoor PM(1.0)']
    analysis_data = joined_data[['Datetime', 'PM1_0_diff'] + predictor_names].dropna()

    X = analysis_data[predictor_names]
    y = analysis_data['PM1_0_diff']
    datetimes = analysis_data['Datetime']
    all_X_list.append(X)

    X_train, X_test, y_train, y_test, dt_train, dt_test = train_test_split(X, y, datetimes, test_size=0.2, random_state=1)

    # ==========================================
    # 3. BASE MODEL (ALL FEATURES)
    # ==========================================
    model_all = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1)
    model_all.fit(X_train, y_train)

    y_pred_test_all = model_all.predict(X_test)
    y_pred_train_all = model_all.predict(X_train)
    r2_test_all = r2_score(y_test, y_pred_test_all)
    r2_train_all = r2_score(y_train, y_pred_train_all)

    # Base Plot 1: Actual vs Predicted (All Features)
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_test_all, color='blue', alpha=0.5, label=f'Test Data (R² = {r2_test_all:.3f})')
    plt.scatter(y_train, y_pred_train_all, color='green', alpha=0.2, marker='.', label=f'Train Data (R² = {r2_train_all:.3f})')
    min_val, max_val = min(y.min(), min(y_pred_test_all)), max(y.max(), max(y_pred_test_all))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')
    plt.xlabel('Actual PM1.0 diff (μg/m³)'); plt.ylabel('Predicted PM1.0 diff (μg/m³)')
    plt.title(f'{indoor_name}: PM1.0 Diff Prediction (All Params)')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    plt.grid(True); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_all_est.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Base Permutation Importance
    perm_imp_all = permutation_importance(model_all, X_test, y_test, n_repeats=5, random_state=1).importances_mean
    all_sorted_idx = perm_imp_all.argsort()

    # Base Plot 2: Full Permutation Imp
    plt.figure(figsize=(12, 10))
    plt.barh(np.array(predictor_names)[all_sorted_idx], perm_imp_all[all_sorted_idx], color='steelblue')
    plt.xlabel('Mean Importance'); plt.title(f'{indoor_name}: Feature Imp PM1.0 (All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_all_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Base SHAP
    test_df = X_test.copy(); test_df['target'] = y_test; test_df['datetime'] = dt_test
    test_df['strata'] = pd.qcut(test_df['target'] + np.random.normal(0, 1e-6, len(test_df)), q=shap_target_bins, labels=False, duplicates='drop')
    shap_sample = test_df.groupby('strata', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(1, shap_swarm_points // shap_target_bins)), random_state=1), include_groups=False)
    X_shap = shap_sample[predictor_names]; y_shap = shap_sample['target'].values; dt_shap = shap_sample['datetime'].values

    kmeans_all = KMeans(n_clusters=min(shap_background_centroids, len(X_train)), random_state=1, n_init='auto').fit(X_train)
    expl_all = shap.TreeExplainer(model_all, data=pd.DataFrame(kmeans_all.cluster_centers_, columns=predictor_names))
    shap_vals_all = expl_all(X_shap)
    mean_abs_shap_all = np.abs(shap_vals_all.values).mean(axis=0)

    # Base Plot 3: Full SHAP Summary
    plt.figure(figsize=(12, 10))
    shap.plots.beeswarm(shap_vals_all, max_display=len(predictor_names), show=False)
    plt.title(f'{indoor_name}: SHAP Summary PM1.0 (All Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_all_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 4. EXTRACT TOP 10s & COMMON FEATURES
    # ==========================================
    # Top 10 Permutation (from All)
    top10_perm_idx = all_sorted_idx[-10:]
    top10_perm_features = np.array(predictor_names)[top10_perm_idx].tolist()

    plt.figure(figsize=(10, 6))
    plt.barh(top10_perm_features, perm_imp_all[top10_perm_idx], color='purple')
    plt.xlabel('Mean Importance'); plt.title(f'{indoor_name}: Top 10 Permutation Importance (Extracted from All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_all_model_top10_perm.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Top 10 SHAP (from All)
    shap_sorted_idx = mean_abs_shap_all.argsort()
    top10_shap_idx = shap_sorted_idx[-10:]
    top10_shap_features = np.array(predictor_names)[top10_shap_idx].tolist()

    plt.figure(figsize=(10, 6))
    plt.barh(top10_shap_features, mean_abs_shap_all[top10_shap_idx], color='darkorange')
    plt.xlabel('Mean Absolute SHAP Value'); plt.title(f'{indoor_name}: Top 10 SHAP Importance (Extracted from All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_all_model_top10_shap.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- IDENTIFY COMMON FEATURES (CONSENSUS) ---
    common_features = list(set(top10_perm_features).intersection(set(top10_shap_features)))

    if common_features:
        # Extract indices and values for the common features
        common_idx = [predictor_names.index(f) for f in common_features]
        common_perm_vals = perm_imp_all[common_idx]
        common_shap_vals = mean_abs_shap_all[common_idx]

        # Sort by SHAP value to make the side-by-side plot flow visually
        sort_order = np.argsort(common_shap_vals)
        c_feat_sorted = np.array(common_features)[sort_order]
        c_perm_sorted = np.array(common_perm_vals)[sort_order]
        c_shap_sorted = np.array(common_shap_vals)[sort_order]

        # Side-by-Side Highlight Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

        ax1.barh(c_feat_sorted, c_perm_sorted, color='purple')
        ax1.set_xlabel('Mean Permutation Importance')
        ax1.set_title('Permutation Scoring')
        ax1.grid(axis='x')

        ax2.barh(c_feat_sorted, c_shap_sorted, color='darkorange')
        ax2.set_xlabel('Mean Absolute SHAP Value')
        ax2.set_title('SHAP Scoring')
        ax2.grid(axis='x')

        fig.suptitle(f'{indoor_name}: Consensus Top Features (Present in both SHAP & Permutation)', fontsize=16)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_common_top_features.png', dpi=300, bbox_inches='tight')
        plt.close()

        # Save to consensus list for CSV output
        for f, p_val, s_val in zip(c_feat_sorted, c_perm_sorted, c_shap_sorted):
            common_features_list.append({
                'Sensor': indoor_name,
                'Feature': f,
                'Permutation_Importance': p_val,
                'SHAP_Value': s_val
            })

    # ==========================================
    # 5. TRAIN NEW SUB-MODELS (TOP 10 ONLY)
    # ==========================================
    # --- MODEL 2: SHAP TOP 10 ---
    X_train_shap10, X_test_shap10 = X_train[top10_shap_features], X_test[top10_shap_features]
    model_shap10 = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1).fit(X_train_shap10, y_train)
    y_pred_test_shap10 = model_shap10.predict(X_test_shap10)
    r2_test_shap10 = r2_score(y_test, y_pred_test_shap10)

    perm_imp_shap10 = permutation_importance(model_shap10, X_test_shap10, y_test, n_repeats=5, random_state=1).importances_mean

    plt.figure(figsize=(10, 6))
    plt.barh(np.array(top10_shap_features)[perm_imp_shap10.argsort()], perm_imp_shap10[perm_imp_shap10.argsort()], color='darkorange')
    plt.title(f'{indoor_name}: Feature Imp (Model trained ON Top 10 SHAP Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_shap10_model_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    bg_shap10 = pd.DataFrame(KMeans(n_clusters=min(shap_background_centroids, len(X_train_shap10)), random_state=1, n_init='auto').fit(X_train_shap10).cluster_centers_, columns=top10_shap_features)
    expl_shap10 = shap.TreeExplainer(model_shap10, data=bg_shap10)
    sv_shap10 = expl_shap10(X_shap[top10_shap_features])

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(sv_shap10, max_display=10, show=False)
    plt.title(f'{indoor_name}: SHAP Summary (Model trained ON Top 10 SHAP Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_shap10_model_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- MODEL 3: PERM TOP 10 ---
    X_train_perm10, X_test_perm10 = X_train[top10_perm_features], X_test[top10_perm_features]
    model_perm10 = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1).fit(X_train_perm10, y_train)
    y_pred_test_perm10 = model_perm10.predict(X_test_perm10)
    r2_test_perm10 = r2_score(y_test, y_pred_test_perm10)

    perm_imp_perm10 = permutation_importance(model_perm10, X_test_perm10, y_test, n_repeats=5, random_state=1).importances_mean

    plt.figure(figsize=(10, 6))
    plt.barh(np.array(top10_perm_features)[perm_imp_perm10.argsort()], perm_imp_perm10[perm_imp_perm10.argsort()], color='purple')
    plt.title(f'{indoor_name}: Feature Imp (Model trained ON Top 10 Perm Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_perm10_model_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    bg_perm10 = pd.DataFrame(KMeans(n_clusters=min(shap_background_centroids, len(X_train_perm10)), random_state=1, n_init='auto').fit(X_train_perm10).cluster_centers_, columns=top10_perm_features)
    expl_perm10 = shap.TreeExplainer(model_perm10, data=bg_perm10)
    sv_perm10 = expl_perm10(X_shap[top10_perm_features])

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(sv_perm10, max_display=10, show=False)
    plt.title(f'{indoor_name}: SHAP Summary (Model trained ON Top 10 Perm Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_perm10_model_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 6. OVERLAY PLOT (ALL vs SHAP10 vs PERM10)
    # ==========================================
    plt.figure(figsize=(12, 10))
    # Base All-Param Data
    plt.scatter(y_train, y_pred_train_all, color='green', alpha=0.1, marker='.', label=f'Train All Params (R² = {r2_train_all:.3f})')
    plt.scatter(y_test, y_pred_test_all, color='blue', alpha=0.3, label=f'Test All Params (R² = {r2_test_all:.3f})')
    # Top 10 SHAP Model Data
    plt.scatter(y_test, y_pred_test_shap10, color='darkorange', alpha=0.5, marker='x', label=f'Test Top 10 SHAP Model (R² = {r2_test_shap10:.3f})')
    # Top 10 Perm Model Data
    plt.scatter(y_test, y_pred_test_perm10, color='purple', alpha=0.5, marker='+', label=f'Test Top 10 Perm Model (R² = {r2_test_perm10:.3f})')

    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')
    plt.xlabel('Actual PM1.0 diff (μg/m³)'); plt.ylabel('Predicted PM1.0 diff (μg/m³)')
    plt.title(f'{indoor_name}: Overlay PM1.0 Prediction Comparison')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=2)
    plt.grid(True); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM1_0_overlay_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 7. POPULATE DICTIONARIES FOR CSV EXPORTS
    # ==========================================
    all_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_all}; all_imp_dict.update(zip(predictor_names, perm_imp_all)); all_importance_list.append(all_imp_dict)
    all_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_all}; all_shap_dict.update(zip(predictor_names, mean_abs_shap_all)); all_shap_summary_list.append(all_shap_dict)

    s10_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_shap10}; s10_imp_dict.update(zip(top10_shap_features, perm_imp_shap10)); top10_shap_model_imp_list.append(s10_imp_dict)
    s10_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_shap10}; s10_shap_dict.update(zip(top10_shap_features, np.abs(sv_shap10.values).mean(axis=0))); top10_shap_model_summary_list.append(s10_shap_dict)

    p10_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_perm10}; p10_imp_dict.update(zip(top10_perm_features, perm_imp_perm10)); top10_perm_model_imp_list.append(p10_imp_dict)
    p10_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_perm10}; p10_shap_dict.update(zip(top10_perm_features, np.abs(sv_perm10.values).mean(axis=0))); top10_perm_model_summary_list.append(p10_shap_dict)

# ==========================================
# 8. EXPORT ALL AGGREGATED TABLES
# ==========================================
print("\nExporting aggregated CSV files...")

if all_importance_list:
    # Original Base Outputs
    pd.DataFrame(all_importance_list).to_csv(f'{output_dir}/allSensor_PM1_0_all_model_imp.csv', index=False)
    pd.DataFrame(all_shap_summary_list).to_csv(f'{output_dir}/allSensor_PM1_0_all_model_shap_meanabs.csv', index=False)

    # New Top 10 Sub-Model Outputs
    pd.DataFrame(top10_shap_model_imp_list).to_csv(f'{output_dir}/allSensor_PM1_0_shap10_model_imp.csv', index=False)
    pd.DataFrame(top10_shap_model_summary_list).to_csv(f'{output_dir}/allSensor_PM1_0_shap10_model_shap_meanabs.csv', index=False)

    pd.DataFrame(top10_perm_model_imp_list).to_csv(f'{output_dir}/allSensor_PM1_0_perm10_model_imp.csv', index=False)
    pd.DataFrame(top10_perm_model_summary_list).to_csv(f'{output_dir}/allSensor_PM1_0_perm10_model_shap_meanabs.csv', index=False)

    # Consensus Features Output
    if common_features_list:
        pd.DataFrame(common_features_list).to_csv(f'{output_dir}/allSensor_PM1_0_common_consensus_features.csv', index=False)

end_time = time.time()
print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")


--- Processing ind01 (70b3d540f40ce423) ---

--- Processing ind02 (70b3d540f40ce429) ---

--- Processing ind03 (70b3d540f40ce435) ---

--- Processing ind04 (70b3d540f40ce430) ---

--- Processing ind06 (70b3d540f40ce43c) ---

--- Processing ind07 (70b3d540f40ce420) ---

--- Processing ind08 (70b3d540f40ce425) ---

--- Processing ind09 (70b3d540f40ce438) ---

--- Processing ind10 (70b3d540f40ce436) ---

--- Processing ind11 (70b3d540f40ce433) ---

--- Processing ind12 (70b3d540f40ce43a) ---

--- Processing ind13 (70b3d540f40ce422) ---

--- Processing ind14 (70b3d540f40ce421) ---

--- Processing ind15 (70b3d540f40ce427) ---

--- Processing ind16 (70b3d540f40ce426) ---

--- Processing ind17 (70b3d540f40ce42f) ---

--- Processing ind18 (70b3d540f40ce434) ---

--- Processing ind19 (70b3d540f40ce424) ---

--- Processing ind20 (70b3d540f40ce42d) ---

--- Processing ind21 (70b3d540f40ce43b) ---

Exporting aggregated CSV files...
Total processing time: 3.86 minutes.


In [ ]:
# ==========================================
# PM 10.0 All Parameters
# ==========================================
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import pvlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
start_time = time.time()
np.random.seed(1)

# Ensure the output directory exists
output_dir = '/content/drive/MyDrive/Data/Results/PM10.0'
os.makedirs(output_dir, exist_ok=True)

# Full Sensor Table from MATLAB
sensor_table = [
    ('ind01', '70b3d540f40ce423'),
    ('ind02', '70b3d540f40ce429'),
    ('ind03', '70b3d540f40ce435'),
    ('ind04', '70b3d540f40ce430'),
    ('ind06', '70b3d540f40ce43c'),
    ('ind07', '70b3d540f40ce420'),
    ('ind08', '70b3d540f40ce425'),
    ('ind09', '70b3d540f40ce438'),
    ('ind10', '70b3d540f40ce436'),
    ('ind11', '70b3d540f40ce433'),
    ('ind12', '70b3d540f40ce43a'),
    ('ind13', '70b3d540f40ce422'),
    ('ind14', '70b3d540f40ce421'),
    ('ind15', '70b3d540f40ce427'),
    ('ind16', '70b3d540f40ce426'),
    ('ind17', '70b3d540f40ce42f'),
    ('ind18', '70b3d540f40ce434'),
    ('ind19', '70b3d540f40ce424'),
    ('ind20', '70b3d540f40ce42d'),
    ('ind21', '70b3d540f40ce43b'),
]

# Runtime settings
shap_swarm_points = 1000
shap_target_bins = 10
shap_background_centroids = 200

# Geographic Coordinates for Solar Angle
lon = 96.70287
lat = 32.96735

# Storage for final CSV outputs
all_importance_list = []
all_shap_summary_list = []
all_shap_values_list = []
all_X_list = []

# Helper function to compute optimal lag via cross-correlation
def get_best_lag(x_series, y_series):
    x = (x_series - x_series.mean()) / x_series.std()
    y = (y_series - y_series.mean()) / y_series.std()
    correlation = signal.correlate(x, y, mode='full')
    lags = signal.correlation_lags(len(x), len(y), mode='full')
    best_lag = lags[np.argmax(correlation)]
    return best_lag

# Column renaming mappings
old_names = [
    'pc0_1','pc0_3','pc0_5', 'pc1_0','pc2_5','pc5_0', 'pc10_0','pm0_1', 'pm0_3', 'pm0_5',
    'pm1_0_outdoorData','pm2_5_outdoorData', 'pm5_0','pm10_0_outdoorData',
    'barrometricPressureBars','airTemperature', 'relativeHumidity','dewPoint',
    'windDirectionMagnetic','windSpeedMetersPerSecond',
    'channelA410nm', 'channelA435nm', 'channelA460nm', 'channelA485nm',
    'channelA510nm', 'channelA535nm', 'channelA560nm', 'channelA585nm',
    'channelA610nm', 'channelA645nm', 'channelA680nm', 'channelA705nm','channelA730nm',
    'channelA760nm', 'channelA810nm', 'channelA860nm', 'channelA900nm', 'channelA940nm',
    'temperature','humidity', 'co2','luminosity','pressure','pm1_0_indoorData',
    'pm2_5_indoorData','pm10_0_indoorData'
]
new_names = [
    'Outdoor PC(0.1)', 'Outdoor PC(0.3)', 'Outdoor PC(0.5)', 'Outdoor PC(1.0)', 'Outdoor PC(2.5)', 'Outdoor PC(5.0)', 'Outdoor PC(10.0)',
    'Outdoor PM(0.1)', 'Outdoor PM(0.3)', 'Outdoor PM(0.5)', 'Outdoor PM(1.0)', 'Outdoor PM(2.5)', 'Outdoor PM(5.0)', 'Outdoor PM(10.0)',
    'Outdoor Pressure', 'Outdoor Temperature', 'Outdoor Humidity', 'Outdoor Dew Point', 'Outdoor Wind Direction', 'Outdoor Wind Speed',
    '410nm', '435nm', '460nm', '485nm', '510nm', '535nm', '560nm', '585nm',
    '610nm', '645nm', '680nm', '705nm', '730nm', '760nm', '810nm', '860nm', '900nm', '940nm',
    'Indoor Temperature', 'Indoor Humidity', 'Indoor CO2', 'Indoor Luminosity', 'Indoor Pressure',
    'Indoor PM(1.0)', 'Indoor PM(2.5)', 'Indoor PM(10.0)'
]
rename_dict = dict(zip(old_names, new_names))

predictor_names = [
    "Outdoor PC(0.1)", "Outdoor PC(0.3)", "Outdoor PC(0.5)", "Outdoor PC(1.0)", "Outdoor PC(2.5)", "Outdoor PC(5.0)", "Outdoor PC(10.0)",
    "Outdoor PM(0.1)", "Outdoor PM(0.3)", "Outdoor PM(0.5)", "Outdoor PM(1.0)", "Outdoor PM(2.5)", "Outdoor PM(5.0)", "Outdoor PM(10.0)",
    "Outdoor Pressure", "Outdoor Temperature", "Outdoor Humidity", "Outdoor Dew Point", "Outdoor Wind Direction", "Outdoor Wind Speed",
    "410nm", "435nm", "460nm", "485nm", "510nm", "535nm", "560nm", "585nm", "610nm", "645nm", "680nm", "705nm",
    "730nm", "760nm", "810nm", "860nm", "900nm", "940nm", "SolarZenithAngle"
]

# ==========================================
# 2. MAIN SENSOR PROCESSING LOOP
# ==========================================
outdoor_file = '/content/drive/MyDrive/Data/MintsData/Indoor/001e064a1520/combinedValo01Data.csv'
outdoor_data = pd.read_csv(outdoor_file, low_memory=False)
outdoor_data['Datetime'] = pd.to_datetime(outdoor_data['Datetime'], errors='coerce')
outdoor_data = outdoor_data.dropna(subset=['Datetime'])

for indoor_name, indoor_id in sensor_table:
    print(f"\n--- Processing {indoor_name} ({indoor_id}) ---")
    indoor_file = f'/content/drive/MyDrive/Data/MintsData/Indoor/{indoor_id}/{indoor_id}_combined.csv'

    if not os.path.exists(indoor_file):
        print(f"File not found: {indoor_file}. Skipping.")
        continue

    indoor_data = pd.read_csv(indoor_file, low_memory=False)
    # Coerce errors to NaT, then drop rows with NaT in Datetime
    indoor_data['Datetime'] = pd.to_datetime(indoor_data['Datetime'], errors='coerce')
    indoor_data = indoor_data.dropna(subset=['Datetime'])

    # Filter PM values <= 200
    mask = (indoor_data['pm1_0'] <= 200) & (indoor_data['pm2_5'] <= 200) & (indoor_data['pm10_0'] <= 200)
    indoor_data = indoor_data[mask]

    # Temporarily join on exact Datetime to find lag (NOW USING PM10.0)
    temp_join = pd.merge(outdoor_data[['Datetime', 'pm10_0']],
                         indoor_data[['Datetime', 'pm10_0']],
                         on='Datetime', how='inner', suffixes=('_left', '_right')).dropna()

    if temp_join.empty:
        print(f"No overlapping data for {indoor_name}. Skipping.")
        continue

    # Calculate lag (NOW USING PM10.0)
    best_lag = get_best_lag(temp_join['pm10_0_left'], temp_join['pm10_0_right'])
    lag_minutes = best_lag * 10

    # Apply lag shift to indoor data
    indoor_data['Datetime'] = indoor_data['Datetime'] + pd.Timedelta(minutes=lag_minutes)

    # Final Join
    # Final Join (Force MATLAB-style suffixes for overlapping columns)
    joined_data = pd.merge(outdoor_data, indoor_data, on='Datetime', how='inner', suffixes=('_outdoorData', '_indoorData')).dropna()
    print(f"Data aligned and joined successfully for {indoor_name}. Rows: {len(joined_data)}")

    # Rename Columns
    joined_data.rename(columns=rename_dict, inplace=True)

    # Calculate Solar Zenith Angle using pvlib
    times_utc = joined_data['Datetime'].dt.tz_localize('UTC')
    times_local = times_utc.dt.tz_convert('Etc/GMT+6')
    solpos = pvlib.solarposition.get_solarposition(times_local, lat, lon)
    joined_data['SolarZenithAngle'] = solpos['zenith'].values

    # Calculate Target (NOW USING PM10.0)
    joined_data['PM10_0_diff'] = joined_data['Outdoor PM(10.0)'] - joined_data['Indoor PM(10.0)']

    # Filter to predictors and drop any remaining NaNs (NOW USING PM10_0_diff)
    analysis_data = joined_data[['Datetime', 'PM10_0_diff'] + predictor_names].dropna()

    X = analysis_data[predictor_names]
    y = analysis_data['PM10_0_diff']
    datetimes = analysis_data['Datetime']
    all_X_list.append(X)

    # Train/Test Split (80/20)
    X_train, X_test, y_train, y_test, dt_train, dt_test = train_test_split(
        X, y, datetimes, test_size=0.2, random_state=1
    )

    # ==========================================
    # 3. TRAIN GPU XGBOOST MODEL
    # ==========================================
    model = xgb.XGBRegressor(
        n_estimators=100,
        tree_method='hist',
        device='cuda', # Forces execution on the H100/T4 GPU
        random_state=1
    )
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)

    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)

    # --- Plot 1: Actual vs Predicted ---
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_test, color='blue', alpha=0.5, label=f'Test Data (R² = {r2_test:.3f})')
    plt.scatter(y_train, y_pred_train, color='green', alpha=0.2, marker='.', label=f'Train Data (R² = {r2_train:.3f})')

    min_val = min(y.min(), min(y_pred_test.min(), y_pred_train.min()))
    max_val = max(y.max(), max(y_pred_test.max(), y_pred_train.max()))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')

    plt.xlabel('Actual PM10.0 difference (μg/m³)')
    plt.ylabel('Predicted PM10.0 difference (μg/m³)')
    plt.title(f'{indoor_name}: PM10.0 Diff Prediction (included)')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_outdoor_est_included.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_outdoor_est_included.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 4. PERMUTATION IMPORTANCE
    # ==========================================
    print("Calculating Feature Importance...")
    perm_importance = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=1)
    imp_means = perm_importance.importances_mean

    # Sort for plotting
    sorted_idx = imp_means.argsort()

    # --- Plot 2: Feature Importance ---
    plt.figure(figsize=(12, 10))
    plt.barh(np.array(predictor_names)[sorted_idx], imp_means[sorted_idx], color='steelblue')
    plt.xlabel('Mean Importance')
    plt.title(f'{indoor_name}: Feature Importance for PM10.0 Diff (included)')
    plt.grid(axis='x')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_outdoor_imp.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_outdoor_imp.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Save importance stats to list
    imp_dict = {'Sensor': indoor_name, 'Lag_Minutes': lag_minutes,
                'MSE_Test': mse_test, 'RMSE_Test': rmse_test, 'R2_Test': r2_test,
                'MSE_Train': mse_train, 'RMSE_Train': rmse_train, 'R2_Train': r2_train}
    for feat, imp in zip(predictor_names, imp_means):
        imp_dict[feat] = imp
    all_importance_list.append(imp_dict)

    # ==========================================
    # 5. SHAP VALUE CALCULATION (GPU Accelerated)
    # ==========================================
    print("Calculating SHAP values...")
    # Stratified Sampling for Swarm Plot
    test_df = X_test.copy()
    test_df['target'] = y_test
    test_df['datetime'] = dt_test

    # Add minor noise to target to prevent duplicate bin edges in qcut
    target_noise = test_df['target'] + np.random.normal(0, 1e-6, len(test_df))
    test_df['strata'] = pd.qcut(target_noise, q=shap_target_bins, labels=False, duplicates='drop')

    # Sample proportionally from bins
    n_per_bin = max(1, shap_swarm_points // shap_target_bins)
    shap_sample = test_df.groupby('strata', group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), n_per_bin), random_state=1),
        include_groups=False
    )

    X_shap = shap_sample[predictor_names]
    y_shap = shap_sample['target'].values
    dt_shap = shap_sample['datetime'].values
    y_shap_pred = model.predict(X_shap)

    # Background KMeans
    kmeans = KMeans(n_clusters=min(shap_background_centroids, len(X_train)), random_state=1, n_init='auto')
    kmeans.fit(X_train)
    background_data = pd.DataFrame(kmeans.cluster_centers_, columns=predictor_names)

    # Execute SHAP
    explainer = shap.TreeExplainer(model, data=background_data)
    shap_values = explainer(X_shap)

    # --- Plot 3: SHAP Summary Swarm Plot ---
    plt.figure(figsize=(12, 10))
    shap.plots.beeswarm(shap_values, max_display=len(predictor_names), show=False)
    plt.title(f'{indoor_name}: SHAP Summary for PM10.0 Diff (included)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_outdoor_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_outdoor_shap_summary.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # Save Mean Absolute SHAP Summary
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    shap_summary_dict = {'Sensor': indoor_name, 'Lag_Minutes': lag_minutes, 'SHAP_Samples': len(X_shap),
                         'MSE_Test': mse_test, 'RMSE_Test': rmse_test, 'R2_Test': r2_test,
                         'MSE_Train': mse_train, 'RMSE_Train': rmse_train, 'R2_Train': r2_train}
    for feat, mean_shap in zip(predictor_names, mean_abs_shap):
        shap_summary_dict[feat] = mean_shap
    all_shap_summary_list.append(shap_summary_dict)

    # Save Long-Format Raw SHAP Values Table
    num_shap_rows = len(X_shap)
    num_predictors = len(predictor_names)

    # Flattening data to match MATLAB's long-format table
    sensor_col = np.repeat(indoor_name, num_shap_rows * num_predictors)
    lag_col = np.repeat(lag_minutes, num_shap_rows * num_predictors)
    query_point_col = np.tile(np.arange(1, num_shap_rows + 1), num_predictors)
    datetime_col = np.tile(dt_shap, num_predictors)
    actual_col = np.tile(y_shap, num_predictors)
    pred_col = np.tile(y_shap_pred, num_predictors)
    predictor_col = np.repeat(predictor_names, num_shap_rows)
    feature_value_col = X_shap.values.T.flatten()
    shap_value_col = shap_values.values.T.flatten()

    long_format_df = pd.DataFrame({
        'Sensor': sensor_col,
        'Lag_Minutes': lag_col,
        'QueryPointIndex': query_point_col,
        'Datetime': datetime_col,
        'Actual_PM10_0_diff': actual_col,
        'Predicted_PM10_0_diff': pred_col,
        'Predictor': predictor_col,
        'FeatureValue': feature_value_col,
        'ShapValue': shap_value_col
    })
    all_shap_values_list.append(long_format_df)

# ==========================================
# 6. EXPORT ALL AGGREGATED TABLES & HEATMAPS
# ==========================================
print("\nExporting aggregated CSV files...")
if all_importance_list:
    df_imp = pd.DataFrame(all_importance_list)
    df_shap = pd.DataFrame(all_shap_summary_list)

    df_imp.to_csv(f'{output_dir}/allSensor_PM10_0_outdoor_importance_included_GPU.csv', index=False)
    df_shap.to_csv(f'{output_dir}/allSensor_PM10_0_outdoor_shap_meanabs_included_GPU.csv', index=False)
    pd.concat(all_shap_values_list, ignore_index=True).to_csv(f'{output_dir}/allSensor_PM10_0_outdoor_shap_values_included_GPU.csv', index=False)

    # ------------------------------------------
    # Correlation Heatmap
    # ------------------------------------------
    if all_X_list:
        print("Generating Correlation Heatmap...")
        X_all = pd.concat(all_X_list, ignore_index=True)
        corr_matrix = X_all.corr()
        corr_matrix.to_csv(f'{output_dir}/predictor_correlation_matrix.csv')

        plt.figure(figsize=(16, 14))
        sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, vmin=-1, vmax=1)
        plt.title('Predictor Correlation Heatmap')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/predictor_correlation_heatmap.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{output_dir}/predictor_correlation_heatmap.pdf', dpi=300, bbox_inches='tight')
        plt.close()

    # ------------------------------------------
    # Normalized Importance Heatmaps
    # ------------------------------------------
    print("Generating Normalized Importance Heatmaps...")

    # Permutation Heatmap
    df_imp_features = df_imp.set_index('Sensor')[predictor_names]
    df_imp_norm = df_imp_features.div(df_imp_features.max(axis=1), axis=0)
    df_imp_norm.to_csv(f'{output_dir}/allSensor_PM10_0_outdoor_importance_normalized.csv')

    plt.figure(figsize=(16, 10))
    sns.heatmap(df_imp_norm, cmap='viridis', annot=False)
    plt.title('Normalized Permutation Importance Across Sensors (PM10.0)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/allSensor_PM10_0_outdoor_importance_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_dir}/allSensor_PM10_0_outdoor_importance_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # SHAP Heatmap
    df_shap_features = df_shap.set_index('Sensor')[predictor_names]
    df_shap_norm = df_shap_features.div(df_shap_features.max(axis=1), axis=0)
    df_shap_norm.to_csv(f'{output_dir}/allSensor_PM10_0_outdoor_shap_normalized.csv')

    plt.figure(figsize=(16, 10))
    sns.heatmap(df_shap_norm, cmap='viridis', annot=False)
    plt.title('Normalized SHAP Importance Across Sensors (PM10.0)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/allSensor_PM10_0_outdoor_shap_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_dir}/allSensor_PM10_0_outdoor_shap_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

end_time = time.time()
print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")


--- Processing ind01 (70b3d540f40ce423) ---
Data aligned and joined successfully for ind01. Rows: 48351
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind02 (70b3d540f40ce429) ---
Data aligned and joined successfully for ind02. Rows: 38710
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind03 (70b3d540f40ce435) ---
Data aligned and joined successfully for ind03. Rows: 45573
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind04 (70b3d540f40ce430) ---
Data aligned and joined successfully for ind04. Rows: 41111
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind06 (70b3d540f40ce43c) ---
Data aligned and joined successfully for ind06. Rows: 51135
Calculating Feature Importance...
Calculating SHAP values...

--- Processing ind07 (70b3d540f40ce420) ---
Data aligned and joined successfully for ind07. Rows: 42573
Calculating Feature Importance...
Calculating SHAP values...

---

In [ ]:
# ==========================================
# PM 10.0 All Parameters + top10
# ==========================================
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import pvlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

# ==========================================
# 1. SETUP AND CONFIGURATION
# ==========================================
start_time = time.time()
np.random.seed(1)

# Ensure the NEW output directory exists
output_dir = '/content/drive/MyDrive/Data/Results/PM10.0/top10_PM10.0'
os.makedirs(output_dir, exist_ok=True)

# Full Sensor Table
sensor_table = [
    ('ind01', '70b3d540f40ce423'),
    ('ind02', '70b3d540f40ce429'),
    ('ind03', '70b3d540f40ce435'),
    ('ind04', '70b3d540f40ce430'),
    ('ind06', '70b3d540f40ce43c'),
    ('ind07', '70b3d540f40ce420'),
    ('ind08', '70b3d540f40ce425'),
    ('ind09', '70b3d540f40ce438'),
    ('ind10', '70b3d540f40ce436'),
    ('ind11', '70b3d540f40ce433'),
    ('ind12', '70b3d540f40ce43a'),
    ('ind13', '70b3d540f40ce422'),
    ('ind14', '70b3d540f40ce421'),
    ('ind15', '70b3d540f40ce427'),
    ('ind16', '70b3d540f40ce426'),
    ('ind17', '70b3d540f40ce42f'),
    ('ind18', '70b3d540f40ce434'),
    ('ind19', '70b3d540f40ce424'),
    ('ind20', '70b3d540f40ce42d'),
    ('ind21', '70b3d540f40ce43b'),
]

shap_swarm_points = 1000
shap_target_bins = 10
shap_background_centroids = 200

lon = 96.70287
lat = 32.96735

# Storage for All Model CSV outputs
all_importance_list = []
all_shap_summary_list = []
all_shap_values_list = []
all_X_list = []

# Storage for Top 10 Sub-Model CSV outputs
top10_shap_model_imp_list = []
top10_shap_model_summary_list = []

top10_perm_model_imp_list = []
top10_perm_model_summary_list = []

# Storage for Common Features Consensus
common_features_list = []

def get_best_lag(x_series, y_series):
    x = (x_series - x_series.mean()) / x_series.std()
    y = (y_series - y_series.mean()) / y_series.std()
    correlation = signal.correlate(x, y, mode='full')
    lags = signal.correlation_lags(len(x), len(y), mode='full')
    return lags[np.argmax(correlation)]

old_names = [
    'pc0_1','pc0_3','pc0_5', 'pc1_0','pc2_5','pc5_0', 'pc10_0','pm0_1', 'pm0_3', 'pm0_5',
    'pm1_0_outdoorData','pm2_5_outdoorData', 'pm5_0','pm10_0_outdoorData',
    'barrometricPressureBars','airTemperature', 'relativeHumidity','dewPoint',
    'windDirectionMagnetic','windSpeedMetersPerSecond',
    'channelA410nm', 'channelA435nm', 'channelA460nm', 'channelA485nm',
    'channelA510nm', 'channelA535nm', 'channelA560nm', 'channelA585nm',
    'channelA610nm', 'channelA645nm', 'channelA680nm', 'channelA705nm','channelA730nm',
    'channelA760nm', 'channelA810nm', 'channelA860nm', 'channelA900nm', 'channelA940nm',
    'temperature','humidity', 'co2','luminosity','pressure','pm1_0_indoorData',
    'pm2_5_indoorData','pm10_0_indoorData'
]
new_names = [
    'Outdoor PC(0.1)', 'Outdoor PC(0.3)', 'Outdoor PC(0.5)', 'Outdoor PC(1.0)', 'Outdoor PC(2.5)', 'Outdoor PC(5.0)', 'Outdoor PC(10.0)',
    'Outdoor PM(0.1)', 'Outdoor PM(0.3)', 'Outdoor PM(0.5)', 'Outdoor PM(1.0)', 'Outdoor PM(2.5)', 'Outdoor PM(5.0)', 'Outdoor PM(10.0)',
    'Outdoor Pressure', 'Outdoor Temperature', 'Outdoor Humidity', 'Outdoor Dew Point', 'Outdoor Wind Direction', 'Outdoor Wind Speed',
    '410nm', '435nm', '460nm', '485nm', '510nm', '535nm', '560nm', '585nm',
    '610nm', '645nm', '680nm', '705nm', '730nm', '760nm', '810nm', '860nm', '900nm', '940nm',
    'Indoor Temperature', 'Indoor Humidity', 'Indoor CO2', 'Indoor Luminosity', 'Indoor Pressure',
    'Indoor PM(1.0)', 'Indoor PM(2.5)', 'Indoor PM(10.0)'
]
rename_dict = dict(zip(old_names, new_names))

predictor_names = [
    "Outdoor PC(0.1)", "Outdoor PC(0.3)", "Outdoor PC(0.5)", "Outdoor PC(1.0)", "Outdoor PC(2.5)", "Outdoor PC(5.0)", "Outdoor PC(10.0)",
    "Outdoor PM(0.1)", "Outdoor PM(0.3)", "Outdoor PM(0.5)", "Outdoor PM(1.0)", "Outdoor PM(2.5)", "Outdoor PM(5.0)", "Outdoor PM(10.0)",
    "Outdoor Pressure", "Outdoor Temperature", "Outdoor Humidity", "Outdoor Dew Point", "Outdoor Wind Direction", "Outdoor Wind Speed",
    "410nm", "435nm", "460nm", "485nm", "510nm", "535nm", "560nm", "585nm", "610nm", "645nm", "680nm", "705nm",
    "730nm", "760nm", "810nm", "860nm", "900nm", "940nm", "SolarZenithAngle"
]

# ==========================================
# 2. MAIN SENSOR PROCESSING LOOP
# ==========================================
outdoor_file = '/content/drive/MyDrive/Data/MintsData/Indoor/001e064a1520/combinedValo01Data.csv'
outdoor_data = pd.read_csv(outdoor_file, low_memory=False)
outdoor_data['Datetime'] = pd.to_datetime(outdoor_data['Datetime'], errors='coerce')
outdoor_data = outdoor_data.dropna(subset=['Datetime'])

for indoor_name, indoor_id in sensor_table:
    print(f"\n--- Processing {indoor_name} ({indoor_id}) ---")
    indoor_file = f'/content/drive/MyDrive/Data/MintsData/Indoor/{indoor_id}/{indoor_id}_combined.csv'

    if not os.path.exists(indoor_file):
        continue

    indoor_data = pd.read_csv(indoor_file, low_memory=False)
    indoor_data['Datetime'] = pd.to_datetime(indoor_data['Datetime'], errors='coerce')
    indoor_data = indoor_data.dropna(subset=['Datetime'])

    mask = (indoor_data['pm1_0'] <= 200) & (indoor_data['pm2_5'] <= 200) & (indoor_data['pm10_0'] <= 200)
    indoor_data = indoor_data[mask]

    temp_join = pd.merge(outdoor_data[['Datetime', 'pm10_0']], indoor_data[['Datetime', 'pm10_0']],
                         on='Datetime', how='inner', suffixes=('_left', '_right')).dropna()

    if temp_join.empty: continue

    lag_minutes = get_best_lag(temp_join['pm10_0_left'], temp_join['pm10_0_right']) * 10
    indoor_data['Datetime'] = indoor_data['Datetime'] + pd.Timedelta(minutes=lag_minutes)

    joined_data = pd.merge(outdoor_data, indoor_data, on='Datetime', how='inner', suffixes=('_outdoorData', '_indoorData')).dropna()
    joined_data.rename(columns=rename_dict, inplace=True)

    times_utc = joined_data['Datetime'].dt.tz_localize('UTC')
    times_local = times_utc.dt.tz_convert('Etc/GMT+6')
    solpos = pvlib.solarposition.get_solarposition(times_local, lat, lon)
    joined_data['SolarZenithAngle'] = solpos['zenith'].values

    joined_data['PM10_0_diff'] = joined_data['Outdoor PM(10.0)'] - joined_data['Indoor PM(10.0)']
    analysis_data = joined_data[['Datetime', 'PM10_0_diff'] + predictor_names].dropna()

    X = analysis_data[predictor_names]
    y = analysis_data['PM10_0_diff']
    datetimes = analysis_data['Datetime']
    all_X_list.append(X)

    X_train, X_test, y_train, y_test, dt_train, dt_test = train_test_split(X, y, datetimes, test_size=0.2, random_state=1)

    # ==========================================
    # 3. BASE MODEL (ALL FEATURES)
    # ==========================================
    model_all = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1)
    model_all.fit(X_train, y_train)

    y_pred_test_all = model_all.predict(X_test)
    y_pred_train_all = model_all.predict(X_train)
    r2_test_all = r2_score(y_test, y_pred_test_all)
    r2_train_all = r2_score(y_train, y_pred_train_all)

    # Base Plot 1: Actual vs Predicted (All Features)
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred_test_all, color='blue', alpha=0.5, label=f'Test Data (R² = {r2_test_all:.3f})')
    plt.scatter(y_train, y_pred_train_all, color='green', alpha=0.2, marker='.', label=f'Train Data (R² = {r2_train_all:.3f})')
    min_val, max_val = min(y.min(), min(y_pred_test_all)), max(y.max(), max(y_pred_test_all))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')
    plt.xlabel('Actual PM10.0 diff (μg/m³)'); plt.ylabel('Predicted PM10.0 diff (μg/m³)')
    plt.title(f'{indoor_name}: PM10.0 Diff Prediction (All Params)')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    plt.grid(True); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_all_est.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Base Permutation Importance
    perm_imp_all = permutation_importance(model_all, X_test, y_test, n_repeats=5, random_state=1).importances_mean
    all_sorted_idx = perm_imp_all.argsort()

    # Base Plot 2: Full Permutation Imp
    plt.figure(figsize=(12, 10))
    plt.barh(np.array(predictor_names)[all_sorted_idx], perm_imp_all[all_sorted_idx], color='steelblue')
    plt.xlabel('Mean Importance'); plt.title(f'{indoor_name}: Feature Imp PM10.0 (All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_all_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Base SHAP
    test_df = X_test.copy(); test_df['target'] = y_test; test_df['datetime'] = dt_test
    test_df['strata'] = pd.qcut(test_df['target'] + np.random.normal(0, 1e-6, len(test_df)), q=shap_target_bins, labels=False, duplicates='drop')
    shap_sample = test_df.groupby('strata', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(1, shap_swarm_points // shap_target_bins)), random_state=1), include_groups=False)
    X_shap = shap_sample[predictor_names]; y_shap = shap_sample['target'].values; dt_shap = shap_sample['datetime'].values

    kmeans_all = KMeans(n_clusters=min(shap_background_centroids, len(X_train)), random_state=1, n_init='auto').fit(X_train)
    expl_all = shap.TreeExplainer(model_all, data=pd.DataFrame(kmeans_all.cluster_centers_, columns=predictor_names))
    shap_vals_all = expl_all(X_shap)
    mean_abs_shap_all = np.abs(shap_vals_all.values).mean(axis=0)

    # Base Plot 3: Full SHAP Summary
    plt.figure(figsize=(12, 10))
    shap.plots.beeswarm(shap_vals_all, max_display=len(predictor_names), show=False)
    plt.title(f'{indoor_name}: SHAP Summary PM10.0 (All Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_all_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 4. EXTRACT TOP 10s & COMMON FEATURES
    # ==========================================
    # Top 10 Permutation (from All)
    top10_perm_idx = all_sorted_idx[-10:]
    top10_perm_features = np.array(predictor_names)[top10_perm_idx].tolist()

    plt.figure(figsize=(10, 6))
    plt.barh(top10_perm_features, perm_imp_all[top10_perm_idx], color='purple')
    plt.xlabel('Mean Importance'); plt.title(f'{indoor_name}: Top 10 Permutation Importance (Extracted from All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_all_model_top10_perm.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Top 10 SHAP (from All)
    shap_sorted_idx = mean_abs_shap_all.argsort()
    top10_shap_idx = shap_sorted_idx[-10:]
    top10_shap_features = np.array(predictor_names)[top10_shap_idx].tolist()

    plt.figure(figsize=(10, 6))
    plt.barh(top10_shap_features, mean_abs_shap_all[top10_shap_idx], color='darkorange')
    plt.xlabel('Mean Absolute SHAP Value'); plt.title(f'{indoor_name}: Top 10 SHAP Importance (Extracted from All Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_all_model_top10_shap.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- IDENTIFY COMMON FEATURES (CONSENSUS) ---
    common_features = list(set(top10_perm_features).intersection(set(top10_shap_features)))

    if common_features:
        # Extract indices and values for the common features
        common_idx = [predictor_names.index(f) for f in common_features]
        common_perm_vals = perm_imp_all[common_idx]
        common_shap_vals = mean_abs_shap_all[common_idx]

        # Sort by SHAP value to make the side-by-side plot flow visually
        sort_order = np.argsort(common_shap_vals)
        c_feat_sorted = np.array(common_features)[sort_order]
        c_perm_sorted = np.array(common_perm_vals)[sort_order]
        c_shap_sorted = np.array(common_shap_vals)[sort_order]

        # Side-by-Side Highlight Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

        ax1.barh(c_feat_sorted, c_perm_sorted, color='purple')
        ax1.set_xlabel('Mean Permutation Importance')
        ax1.set_title('Permutation Scoring')
        ax1.grid(axis='x')

        ax2.barh(c_feat_sorted, c_shap_sorted, color='darkorange')
        ax2.set_xlabel('Mean Absolute SHAP Value')
        ax2.set_title('SHAP Scoring')
        ax2.grid(axis='x')

        fig.suptitle(f'{indoor_name}: Consensus Top Features (Present in both SHAP & Permutation)', fontsize=16)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_common_top_features.png', dpi=300, bbox_inches='tight')
        plt.close()

        # Save to consensus list for CSV output
        for f, p_val, s_val in zip(c_feat_sorted, c_perm_sorted, c_shap_sorted):
            common_features_list.append({
                'Sensor': indoor_name,
                'Feature': f,
                'Permutation_Importance': p_val,
                'SHAP_Value': s_val
            })

    # ==========================================
    # 5. TRAIN NEW SUB-MODELS (TOP 10 ONLY)
    # ==========================================
    # --- MODEL 2: SHAP TOP 10 ---
    X_train_shap10, X_test_shap10 = X_train[top10_shap_features], X_test[top10_shap_features]
    model_shap10 = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1).fit(X_train_shap10, y_train)
    y_pred_test_shap10 = model_shap10.predict(X_test_shap10)
    r2_test_shap10 = r2_score(y_test, y_pred_test_shap10)

    perm_imp_shap10 = permutation_importance(model_shap10, X_test_shap10, y_test, n_repeats=5, random_state=1).importances_mean

    plt.figure(figsize=(10, 6))
    plt.barh(np.array(top10_shap_features)[perm_imp_shap10.argsort()], perm_imp_shap10[perm_imp_shap10.argsort()], color='darkorange')
    plt.title(f'{indoor_name}: Feature Imp (Model trained ON Top 10 SHAP Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_shap10_model_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    bg_shap10 = pd.DataFrame(KMeans(n_clusters=min(shap_background_centroids, len(X_train_shap10)), random_state=1, n_init='auto').fit(X_train_shap10).cluster_centers_, columns=top10_shap_features)
    expl_shap10 = shap.TreeExplainer(model_shap10, data=bg_shap10)
    sv_shap10 = expl_shap10(X_shap[top10_shap_features])

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(sv_shap10, max_display=10, show=False)
    plt.title(f'{indoor_name}: SHAP Summary (Model trained ON Top 10 SHAP Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_shap10_model_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # --- MODEL 3: PERM TOP 10 ---
    X_train_perm10, X_test_perm10 = X_train[top10_perm_features], X_test[top10_perm_features]
    model_perm10 = xgb.XGBRegressor(n_estimators=100, tree_method='hist', device='cuda', random_state=1).fit(X_train_perm10, y_train)
    y_pred_test_perm10 = model_perm10.predict(X_test_perm10)
    r2_test_perm10 = r2_score(y_test, y_pred_test_perm10)

    perm_imp_perm10 = permutation_importance(model_perm10, X_test_perm10, y_test, n_repeats=5, random_state=1).importances_mean

    plt.figure(figsize=(10, 6))
    plt.barh(np.array(top10_perm_features)[perm_imp_perm10.argsort()], perm_imp_perm10[perm_imp_perm10.argsort()], color='purple')
    plt.title(f'{indoor_name}: Feature Imp (Model trained ON Top 10 Perm Params)')
    plt.grid(axis='x'); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_perm10_model_imp.png', dpi=300, bbox_inches='tight')
    plt.close()

    bg_perm10 = pd.DataFrame(KMeans(n_clusters=min(shap_background_centroids, len(X_train_perm10)), random_state=1, n_init='auto').fit(X_train_perm10).cluster_centers_, columns=top10_perm_features)
    expl_perm10 = shap.TreeExplainer(model_perm10, data=bg_perm10)
    sv_perm10 = expl_perm10(X_shap[top10_perm_features])

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(sv_perm10, max_display=10, show=False)
    plt.title(f'{indoor_name}: SHAP Summary (Model trained ON Top 10 Perm Params)')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_perm10_model_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 6. OVERLAY PLOT (ALL vs SHAP10 vs PERM10)
    # ==========================================
    plt.figure(figsize=(12, 10))
    # Base All-Param Data
    plt.scatter(y_train, y_pred_train_all, color='green', alpha=0.1, marker='.', label=f'Train All Params (R² = {r2_train_all:.3f})')
    plt.scatter(y_test, y_pred_test_all, color='blue', alpha=0.3, label=f'Test All Params (R² = {r2_test_all:.3f})')
    # Top 10 SHAP Model Data
    plt.scatter(y_test, y_pred_test_shap10, color='darkorange', alpha=0.5, marker='x', label=f'Test Top 10 SHAP Model (R² = {r2_test_shap10:.3f})')
    # Top 10 Perm Model Data
    plt.scatter(y_test, y_pred_test_perm10, color='purple', alpha=0.5, marker='+', label=f'Test Top 10 Perm Model (R² = {r2_test_perm10:.3f})')

    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 Line')
    plt.xlabel('Actual PM10.0 diff (μg/m³)'); plt.ylabel('Predicted PM10.0 diff (μg/m³)')
    plt.title(f'{indoor_name}: Overlay PM10.0 Prediction Comparison')
    plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=2)
    plt.grid(True); plt.tight_layout()
    plt.savefig(f'{output_dir}/{indoor_name}_PM10_0_overlay_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # 7. POPULATE DICTIONARIES FOR CSV EXPORTS
    # ==========================================
    all_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_all}; all_imp_dict.update(zip(predictor_names, perm_imp_all)); all_importance_list.append(all_imp_dict)
    all_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_all}; all_shap_dict.update(zip(predictor_names, mean_abs_shap_all)); all_shap_summary_list.append(all_shap_dict)

    s10_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_shap10}; s10_imp_dict.update(zip(top10_shap_features, perm_imp_shap10)); top10_shap_model_imp_list.append(s10_imp_dict)
    s10_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_shap10}; s10_shap_dict.update(zip(top10_shap_features, np.abs(sv_shap10.values).mean(axis=0))); top10_shap_model_summary_list.append(s10_shap_dict)

    p10_imp_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_perm10}; p10_imp_dict.update(zip(top10_perm_features, perm_imp_perm10)); top10_perm_model_imp_list.append(p10_imp_dict)
    p10_shap_dict = {'Sensor': indoor_name, 'R2_Test': r2_test_perm10}; p10_shap_dict.update(zip(top10_perm_features, np.abs(sv_perm10.values).mean(axis=0))); top10_perm_model_summary_list.append(p10_shap_dict)

# ==========================================
# 8. EXPORT ALL AGGREGATED TABLES
# ==========================================
print("\nExporting aggregated CSV files...")

if all_importance_list:
    # Original Base Outputs
    pd.DataFrame(all_importance_list).to_csv(f'{output_dir}/allSensor_PM10_0_all_model_imp.csv', index=False)
    pd.DataFrame(all_shap_summary_list).to_csv(f'{output_dir}/allSensor_PM10_0_all_model_shap_meanabs.csv', index=False)

    # New Top 10 Sub-Model Outputs
    pd.DataFrame(top10_shap_model_imp_list).to_csv(f'{output_dir}/allSensor_PM10_0_shap10_model_imp.csv', index=False)
    pd.DataFrame(top10_shap_model_summary_list).to_csv(f'{output_dir}/allSensor_PM10_0_shap10_model_shap_meanabs.csv', index=False)

    pd.DataFrame(top10_perm_model_imp_list).to_csv(f'{output_dir}/allSensor_PM10_0_perm10_model_imp.csv', index=False)
    pd.DataFrame(top10_perm_model_summary_list).to_csv(f'{output_dir}/allSensor_PM10_0_perm10_model_shap_meanabs.csv', index=False)

    # Consensus Features Output
    if common_features_list:
        pd.DataFrame(common_features_list).to_csv(f'{output_dir}/allSensor_PM10_0_common_consensus_features.csv', index=False)

end_time = time.time()
print(f"Total processing time: {(end_time - start_time) / 60:.2f} minutes.")


--- Processing ind01 (70b3d540f40ce423) ---

--- Processing ind02 (70b3d540f40ce429) ---

--- Processing ind03 (70b3d540f40ce435) ---

--- Processing ind04 (70b3d540f40ce430) ---

--- Processing ind06 (70b3d540f40ce43c) ---

--- Processing ind07 (70b3d540f40ce420) ---

--- Processing ind08 (70b3d540f40ce425) ---

--- Processing ind09 (70b3d540f40ce438) ---

--- Processing ind10 (70b3d540f40ce436) ---

--- Processing ind11 (70b3d540f40ce433) ---

--- Processing ind12 (70b3d540f40ce43a) ---

--- Processing ind13 (70b3d540f40ce422) ---

--- Processing ind14 (70b3d540f40ce421) ---

--- Processing ind15 (70b3d540f40ce427) ---

--- Processing ind16 (70b3d540f40ce426) ---

--- Processing ind17 (70b3d540f40ce42f) ---

--- Processing ind18 (70b3d540f40ce434) ---

--- Processing ind19 (70b3d540f40ce424) ---

--- Processing ind20 (70b3d540f40ce42d) ---

--- Processing ind21 (70b3d540f40ce43b) ---

Exporting aggregated CSV files...
Total processing time: 5.15 minutes.
